# Finale Tabellen und Abbildungen

Erzeugt zentrale Tabellen, Detailtabellen, Ergebnisabbildungen und Exportprotokolle aus vorhandenen Ergebnisdateien.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
__file__ = str(PROJECT_ROOT / "exports" / "notebooks" / "01_finale_tabellen_und_abbildungen_notebook.py")
print(f"Projektordner: {PROJECT_ROOT}")


## Code


In [ ]:
"""Generate thesis tables and figures for the TikTok influencer analysis.

The script is intentionally defensive: it detects available columns, merges
video-level comment results when present, skips unavailable outputs, and writes a
missing-variable report instead of failing on incomplete data.
"""

from __future__ import annotations

import math
import textwrap
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats


COLOR_VI = "#2A9D8F"
COLOR_RI = "#B85C7A"
COLOR_HUMAN_EVAL_PRIMARY = "#2A9D8F"
COLOR_HUMAN_EVAL_SECONDARY = "#B85C7A"
GROUP_ORDER = ["VI", "RI"]
GROUP_LABELS = {
    "VI": "Virtuelle Influencerinnen",
    "RI": "Menschliche Influencerinnen",
}
PALETTE = {"VI": COLOR_VI, "RI": COLOR_RI}


@dataclass(frozen=True)
class FeatureSpec:
    label: str
    category: str
    aliases: tuple[str, ...]
    method: str
    aggregation: str
    interpretation: str


VISUAL_FEATURES: list[FeatureSpec] = [
    FeatureSpec(
        "Videolänge",
        "Strukturale Inszenierung und visuelle Dynamik",
        ("video_duration_seconds", "duration_seconds", "cuts__video_duration_seconds_est"),
        "Metadaten/Videoanalyse",
        "Dauer pro Video",
        "Längere Werte stehen für längere Videos.",
    ),
    FeatureSpec(
        "Cut Count",
        "Strukturale Inszenierung und visuelle Dynamik",
        ("cut_count", "cuts__cut_count"),
        "Frame-Differenzanalyse mit Schwelle 25.0 und Mindestabstand 1 Frame",
        "Anzahl erkannter Schnitte pro Video",
        "Höhere Werte deuten auf dynamischere Montage hin.",
    ),
    FeatureSpec(
        "Cuts pro Sekunde",
        "Strukturale Inszenierung und visuelle Dynamik",
        ("cuts_per_second", "cuts__cuts_per_second"),
        "Frame-Differenzanalyse mit Schwelle 25.0 und Mindestabstand 1 Frame",
        "Schnitte pro Videosekunde",
        "Höhere Werte deuten auf schnellere visuelle Dynamik hin.",
    ),
    FeatureSpec(
        "Sekunden pro Cut",
        "Strukturale Inszenierung und visuelle Dynamik",
        ("seconds_per_cut",),
        "Abgeleitet aus Videolänge und Cut Count",
        "Videolänge geteilt durch Anzahl erkannter Schnitte; nur für Videos mit mindestens einem erkannten Schnitt definiert",
        "Höhere Werte deuten auf langsamere Schnittfrequenz hin.",
    ),
    FeatureSpec(
        "Szenenvielfalt",
        "Strukturale Inszenierung und visuelle Dynamik",
        ("scene_unique_labels", "scene_classification__scene_unique_labels"),
        "Places365 ResNet18-Szenenklassifikation",
        "Anzahl unterschiedlicher Szenenlabels pro Video",
        "Höhere Werte deuten auf größere Szenenvariation hin.",
    ),
    FeatureSpec(
        "Kameradistanzvariation",
        "Strukturale Inszenierung und visuelle Dynamik",
        ("camera_distance_unique_labels", "camera_distance__camera_distance_unique_labels"),
        "DINOv2 Shot-Scale-Klassifikation (pszemraj/dinov2-small-film-shot-classifier)",
        "Anzahl unterschiedlicher Kameradistanzlabels pro Video",
        "Höhere Werte deuten auf stärkere Variation der Einstellungsgrößen hin.",
    ),
    FeatureSpec(
        "Helligkeit",
        "Bildtechnische Qualität",
        ("brightness_index", "brightness_contrast__brightness_index"),
        "OpenCV: Mittelwert der Graustufenhelligkeit je Frame",
        "Mittelwert über gesampelte Frames auf Videoebene",
        "Höhere Werte stehen für hellere Bildgestaltung.",
    ),
    FeatureSpec(
        "Kontrast",
        "Bildtechnische Qualität",
        ("contrast_index", "brightness_contrast__contrast_index"),
        "OpenCV: Standardabweichung der Graustufenhelligkeit je Frame",
        "Mittelwert über gesampelte Frames auf Videoebene",
        "Höhere Werte stehen für stärkere Helligkeitskontraste.",
    ),
    FeatureSpec(
        "Sättigung",
        "Bildtechnische Qualität",
        ("saturation_index", "saturation__saturation_index"),
        "OpenCV: Mittelwert des HSV-Sättigungskanals je Frame",
        "Mittelwert über gesampelte Frames auf Videoebene",
        "Höhere Werte stehen für intensivere Farbsättigung.",
    ),
    FeatureSpec(
        "Schärfe",
        "Bildtechnische Qualität",
        ("sharpness_laplacian_var", "visual_sharpness__sharpness_laplacian_mean"),
        "OpenCV: Varianz des Laplacian-Filters je Frame",
        "Mittelwert über gesampelte Frames auf Videoebene",
        "Höhere Werte stehen für stärkere Kanten-/Schärfeausprägung.",
    ),
    FeatureSpec(
        "Aesthetic Score",
        "Ästhetik und visuelle Optimierung",
        ("aesthetic_score_mean", "aesthetic_quality__aesthetic_quality_score"),
        "Aesthetics Predictor ViT-B/32 (shunk031/aesthetics-predictor-v1-vit-base-patch32)",
        "Mittelwert der Frame-Scores auf Videoebene",
        "Höhere Werte stehen für höhere geschätzte ästhetische Qualität.",
    ),
    FeatureSpec(
        "Beauty Score",
        "Ästhetik und visuelle Optimierung",
        ("beauty_score_mean", "beauty_scoring__beauty_score_mean"),
        "OpenCV Haar-Face-Detection und ResNeXt50-Caffe Beauty-Modell",
        "Mittelwert der erkannten Face-Scores auf Videoebene",
        "Höhere Werte stehen für höhere geschätzte Schönheitsbewertung.",
    ),
    FeatureSpec(
        "Beauty Score SD",
        "Ästhetik und visuelle Optimierung",
        ("beauty_score_std", "beauty_scoring__beauty_score_std"),
        "OpenCV Haar-Face-Detection und ResNeXt50-Caffe Beauty-Modell",
        "Standardabweichung der erkannten Face-Scores auf Videoebene",
        "Höhere Werte stehen für stärkere Variation der Bewertung im Video.",
    ),
    FeatureSpec(
        "Hautglättung",
        "Ästhetik und visuelle Optimierung",
        ("skin_smoothness_highpass_index", "skin_smoothness__skin_smoothness_highpass_index"),
        "Gesichtsbasierte Highpass-Texturmetrik; Index = 1 / (1 + Highpass-Varianz)",
        "Mittelwert der erkannten Face-/Skin-Frame-Scores auf Videoebene",
        "Höhere Werte deuten auf geringere Highpass-Textur und damit glattere Hautdarstellung hin.",
    ),
    FeatureSpec(
        "Face Pitch",
        "Emotion und Präsenz",
        ("face_pitch_mean", "angle_face_orientation__face_pitch_mean"),
        "MediaPipe FaceDetection-Keypoints; approximierte Pitch-Winkel in Grad",
        "Mittelwert erkannter Face-Frames auf Videoebene",
        "Positive Werte entsprechen im verwendeten Schema einer stärkeren Down-Orientierung, negative Werte einer Up-Orientierung.",
    ),
    FeatureSpec(
        "Face Yaw",
        "Emotion und Präsenz",
        ("face_yaw_mean", "angle_face_orientation__face_yaw_mean"),
        "MediaPipe FaceDetection-Keypoints; approximierte Yaw-Winkel in Grad",
        "Mittelwert erkannter Face-Frames auf Videoebene",
        "Positive Werte entsprechen im verwendeten Schema einer stärkeren Right-Orientierung, negative Werte einer Left-Orientierung.",
    ),
    FeatureSpec(
        "Pose Confidence",
        "Emotion und Präsenz",
        ("pose_confidence_mean", "body_pose__pose_confidence"),
        "SwinV2 Pose-Klassifikation (Harsha901/swinv2-tiny-human-pose-classification-model)",
        "Mittelwert der Klassifikationskonfidenzen über gesampelte Frames",
        "Höhere Werte stehen für sicherere Pose-Annotationen.",
    ),
    FeatureSpec(
        "Personenanzahl",
        "Emotion und Präsenz",
        ("person_count_mean", "structural_personen_anzahl__personen_anzahl"),
        "Kombination aus RetinaFace-Gesichtserkennung und OpenCV-HOG-Personendetektor",
        "Mittelwert der geschätzten Personenzahl über gesampelte Frames",
        "Höhere Werte stehen für mehr erkannte Personen im Video.",
    ),
]

TEXT_FEATURES: list[FeatureSpec] = [
    FeatureSpec(
        "Caption-Länge Zeichen",
        "Textuelle Merkmale",
        ("caption_length_chars",),
        "Textlängenmessung",
        "Anzahl Zeichen pro Caption",
        "Höhere Werte stehen für längere Captions.",
    ),
    FeatureSpec(
        "Caption-Länge Wörter",
        "Textuelle Merkmale",
        ("caption_length_words",),
        "Textlängenmessung",
        "Anzahl Wörter pro Caption",
        "Höhere Werte stehen für längere Captions.",
    ),
    FeatureSpec(
        "Caption-Sentiment-Index",
        "Textuelle Merkmale",
        ("caption_sentiment_index", "caption_sentiment_score"),
        "RoBERTa-Sentimentmodell cardiffnlp/twitter-roberta-base-sentiment-latest",
        "Score pro Caption auf Videoebene",
        "Positive Werte deuten auf positiveres Caption-Sentiment hin.",
    ),
    FeatureSpec(
        "Kommentar-Sentiment-Index",
        "Textuelle Merkmale",
        ("comment_sentiment_index_mean", "sentiment_index_mean", "comment_sentiment_mean"),
        "RoBERTa-Sentimentmodell cardiffnlp/twitter-roberta-base-sentiment-latest",
        "Mittelwert der Kommentare pro Video",
        "Positive Werte deuten auf positiveres Kommentar-Sentiment hin.",
    ),
    FeatureSpec(
        "Kommentar-Emotionsvielfalt",
        "Textuelle Merkmale",
        ("comment_emotion_unique_labels_mean", "emotion_unique_labels_mean"),
        "GoEmotions-Modell SamLowe/roberta-base-go_emotions mit Schwelle 0.10",
        "Mittlere Anzahl der pro Kommentar aktivierten Emotionslabels pro Video",
        "Höhere Werte deuten auf vielfältigere Kommentar-Emotionen hin.",
    ),
    FeatureSpec(
        "Topic-Vielfalt",
        "Textuelle Merkmale",
        ("topic_unique_labels",),
        "Sentence-Transformer all-MiniLM-L6-v2 Embeddings und MiniBatchKMeans-Clustering",
        "Anzahl unterschiedlicher Topic-Labels pro Video",
        "Höhere Werte deuten auf größere thematische Vielfalt in Kommentaren hin.",
    ),
]

LABEL_FEATURES = [
    ("Caption-Sentiment", ("caption_sentiment_label",)),
    ("Kommentar-Sentiment", ("dominant_comment_sentiment", "comment_sentiment_label")),
    ("Kommentar-Emotion", ("dominant_comment_emotion", "comment_emotion_label")),
    ("Dominante Topics", ("dominant_topic_label", "topic_label", "topic_name", "topic")),
]

APPENDIX_LABEL_PLOTS = [
    ("Kamera-Distanz-Labels", ("camera_distance__camera_distance_label", "camera_distance_top1_label"), "app_kamera_distanz_labels"),
    ("Dominante Emotion", ("face_emotion__emotion_major_beit_readable", "dominant_emotion"), "app_dominante_emotion"),
    ("Face Orientation", ("angle_face_orientation__angle_face_orientation", "face_orientation_label"), "app_face_orientation"),
    ("Pose-Labels", ("body_pose__pose_orientation",), "app_pose_labels"),
]


@dataclass(frozen=True)
class ReportingFeatureSpec:
    raw_name: str
    category: str
    subcategory: str
    label: str
    hypothesis: str
    include_main: bool
    include_appendix: bool
    technical: bool
    order: int
    aliases: tuple[str, ...] = ()
    kind: str = "numeric"


CATEGORY_FILES = {
    "Strukturale Inszenierung und visuelle Dynamik": ("h1", "structural_dynamics"),
    "Bildtechnische Qualität": ("h2", "image_quality"),
    "Ästhetik und visuelle Optimierung": ("h3", "aesthetics_visual_optimization"),
    "Emotion und Präsenz": ("h4", "emotion_presence"),
    "Textuelle Merkmale": ("h5", "textual_features"),
}


def R(
    raw_name: str,
    category: str,
    label: str,
    order: int,
    *,
    hypothesis: str,
    subcategory: str = "",
    main: bool = False,
    technical: bool = False,
    aliases: tuple[str, ...] = (),
    kind: str = "numeric",
) -> ReportingFeatureSpec:
    """Short constructor for the central reporting configuration."""
    return ReportingFeatureSpec(
        raw_name=raw_name,
        category=category,
        subcategory=subcategory or category,
        label=label,
        hypothesis=hypothesis,
        include_main=main,
        include_appendix=True,
        technical=technical,
        order=order,
        aliases=aliases,
        kind=kind,
    )


STRUCTURAL = "Strukturale Inszenierung und visuelle Dynamik"
IMAGE_QUALITY = "Bildtechnische Qualität"
AESTHETICS = "Ästhetik und visuelle Optimierung"
EMOTION_PRESENCE = "Emotion und Präsenz"
TEXTUAL = "Textuelle Merkmale"

# Zentrale Steuerstelle für die Ergebnisdarstellung der Arbeit. Variablen für den Haupttext
# sind bewusst explizit gepflegt. Die Detailauswertung erkennt zusätzlich berechnete
# Modellspalten, die noch nicht gelistet sind, damit neu ergänzte technische Merkmale
# automatisch dokumentiert werden.
REPORTING_FEATURES: list[ReportingFeatureSpec] = [
    R("video_duration_seconds", STRUCTURAL, "Videolänge", 101, hypothesis="H1", main=True),
    R("cuts__cut_count", STRUCTURAL, "Cut Count", 102, hypothesis="H1", main=True, aliases=("cut_count",)),
    R("cuts__cuts_per_second", STRUCTURAL, "Cuts pro Sekunde", 103, hypothesis="H1", main=True, aliases=("cuts_per_second",)),
    R("seconds_per_cut", STRUCTURAL, "Sekunden pro Cut", 104, hypothesis="H1", main=True),
    R("scene_classification__scene_unique_labels", STRUCTURAL, "Szenenvielfalt", 105, hypothesis="H1", main=True, aliases=("scene_unique_labels",)),
    R("camera_distance__camera_distance_unique_labels", STRUCTURAL, "Kameradistanzvariation", 106, hypothesis="H1", main=True, aliases=("camera_distance_unique_labels",)),
    R("camera_stability__stability_index", STRUCTURAL, "Stabilitätsindex", 107, hypothesis="H1", main=True, aliases=("stability_index",)),
    R("cuts__frames_scanned", STRUCTURAL, "Analysierte Frames", 151, hypothesis="H1", technical=True, aliases=("frames_scanned",)),
    R("cuts__video_fps", STRUCTURAL, "Bildrate des Videos", 152, hypothesis="H1", technical=True, aliases=("video_fps",)),
    R("cuts__cuts_per_frame", STRUCTURAL, "Cuts pro Frame", 153, hypothesis="H1", technical=True, aliases=("cuts_per_frame",)),
    R("cuts__video_duration_seconds_est", STRUCTURAL, "Geschätzte Videolänge", 154, hypothesis="H1", technical=True, aliases=("video_duration_seconds_est",)),
    R("camera_distance__camera_distance_confidence", STRUCTURAL, "Konfidenz der Kameradistanzklassifikation", 155, hypothesis="H1", technical=True),
    R("camera_distance__processed_frame_count", STRUCTURAL, "Ausgewertete Frames der Kameradistanzanalyse", 156, hypothesis="H1", technical=True),
    R("scene_classification__scene_top1_confidence", STRUCTURAL, "Konfidenz der Szenenklassifikation", 157, hypothesis="H1", technical=True),
    R("camera_stability__optical_flow_magnitude_mean", STRUCTURAL, "Durchschnittliche Bewegungsstärke", 158, hypothesis="H1", technical=True),
    R("camera_stability__optical_flow_magnitude_std", STRUCTURAL, "Streuung der Bewegungsstärke", 159, hypothesis="H1", technical=True),
    R("camera_stability__processed_frame_pairs", STRUCTURAL, "Ausgewertete Frame-Paare", 160, hypothesis="H1", technical=True),
    R("brightness_contrast__brightness_index", IMAGE_QUALITY, "Helligkeit", 201, hypothesis="H2", main=True, aliases=("brightness_index",)),
    R("brightness_contrast__contrast_index", IMAGE_QUALITY, "Kontrast", 202, hypothesis="H2", main=True, aliases=("contrast_index",)),
    R("saturation__saturation_index", IMAGE_QUALITY, "Sättigung", 203, hypothesis="H2", main=True, aliases=("saturation_index",)),
    R("visual_sharpness__sharpness_laplacian_mean", IMAGE_QUALITY, "Durchschnittliche Schärfe", 204, hypothesis="H2", main=True, aliases=("sharpness_laplacian_mean", "sharpness_laplacian_var")),
    R("visual_sharpness__sharpness_laplacian_std", IMAGE_QUALITY, "Streuung der Bildschärfe", 205, hypothesis="H2", main=True, aliases=("sharpness_laplacian_std",)),
    R("visual_sharpness__processed_frame_count", IMAGE_QUALITY, "Ausgewertete Frames der Schärfeanalyse", 251, hypothesis="H2", technical=True),
    R("aesthetic_quality__aesthetic_quality_score", AESTHETICS, "Aesthetic Score", 301, hypothesis="H3", main=True, aliases=("aesthetic_quality_score", "aesthetic_score_mean")),
    R("beauty_scoring__beauty_score_mean", AESTHETICS, "Beauty Score", 302, hypothesis="H3", main=True, aliases=("beauty_score_mean",)),
    R("beauty_scoring__beauty_score_std", AESTHETICS, "Beauty Score SD", 303, hypothesis="H3", main=True, aliases=("beauty_score_std",)),
    R("skin_smoothness__skin_smoothness_highpass_index", AESTHETICS, "Hautglättung Highpass", 304, hypothesis="H3", main=True, aliases=("skin_smoothness_highpass_index",)),
    R("skin_smoothness__skin_smoothness_dog_index", AESTHETICS, "Hautglättung DoG", 305, hypothesis="H3", main=True, aliases=("skin_smoothness_dog_index",)),
    R("visual_filter__filter_strength_prob", AESTHETICS, "Wahrscheinlichkeit für Filtereinsatz", 306, hypothesis="H3", main=True, aliases=("filter_strength_prob",)),
    R("visual_filter__filter_strength_std", AESTHETICS, "Streuung der Filterstärke", 307, hypothesis="H3", main=True, aliases=("filter_strength_std",)),
    R("aesthetic_quality__aesthetic_quality_scored_frames", AESTHETICS, "Anzahl ästhetisch bewerteter Frames", 351, hypothesis="H3", technical=True),
    R("beauty_scoring__beauty_detected_face_frames", AESTHETICS, "Anzahl bewertbarer Gesichtsframes", 352, hypothesis="H3", technical=True),
    R("skin_smoothness__detected_skin_face_frames", AESTHETICS, "Anzahl auswertbarer Hautframes", 353, hypothesis="H3", technical=True),
    R("skin_smoothness__skin_selected_sigma", AESTHETICS, "Gewählter Sigma-Wert der Hautanalyse", 354, hypothesis="H3", technical=True),
    R("skin_smoothness__skin_texture_laplacian_var", AESTHETICS, "Feinstruktur der Hauttextur", 355, hypothesis="H3", technical=True),
    R("skin_smoothness__skin_texture_highpass_var", AESTHETICS, "Hochfrequenztextur der Haut", 356, hypothesis="H3", technical=True),
    R("skin_smoothness__skin_texture_dog_var", AESTHETICS, "Texturkontrast der Haut", 357, hypothesis="H3", technical=True),
    R("visual_filter__processed_frame_count", AESTHETICS, "Ausgewertete Frames der Filteranalyse", 358, hypothesis="H3", technical=True),
    R("face_emotion__emotion_major_beit_confidence", EMOTION_PRESENCE, "Emotionssicherheit", 401, hypothesis="H4", subcategory="Gesichtsemotion", main=True),
    R("face_emotion__emotion_unique_labels", EMOTION_PRESENCE, "Emotionsvielfalt", 402, hypothesis="H4", subcategory="Gesichtsemotion", main=True, aliases=("emotion_unique_labels",)),
    R("face_emotion__emotion_major_beit_readable", EMOTION_PRESENCE, "Dominante Gesichtsemotion", 403, hypothesis="H4", subcategory="Gesichtsemotion", main=True, kind="categorical"),
    R("face_emotion__detected_emotion_frames", EMOTION_PRESENCE, "Anzahl emotional auswertbarer Frames", 404, hypothesis="H4", subcategory="Gesichtsemotion", technical=True),
    R("body_pose__pose_confidence", EMOTION_PRESENCE, "Pose Confidence", 405, hypothesis="H4", subcategory="Körperpose", main=True, aliases=("pose_confidence", "pose_confidence_mean")),
    R("body_pose__pose_orientation", EMOTION_PRESENCE, "Dominante Körperpose", 406, hypothesis="H4", subcategory="Körperpose", main=True, kind="categorical"),
    R("body_pose__detected_pose_frames", EMOTION_PRESENCE, "Anzahl erkannter Pose-Frames", 407, hypothesis="H4", subcategory="Körperpose", technical=True),
    R("structural_personen_anzahl__personen_anzahl", EMOTION_PRESENCE, "Personenanzahl", 408, hypothesis="H4", subcategory="Personenanzahl", main=True, aliases=("personen_anzahl", "person_count_mean")),
    R("structural_personen_anzahl__personen_anzahl_max", EMOTION_PRESENCE, "Maximale Personenanzahl", 409, hypothesis="H4", subcategory="Personenanzahl", main=True, aliases=("personen_anzahl_max",)),
    R("structural_personen_anzahl__detected_person_frames", EMOTION_PRESENCE, "Anzahl erkannter Personenframes", 410, hypothesis="H4", subcategory="Personenanzahl", technical=True),
    R("angle_face_orientation__face_pitch_mean", EMOTION_PRESENCE, "Face Pitch", 411, hypothesis="H4", subcategory="Gesichtsorientierung", main=True, aliases=("face_pitch_mean",)),
    R("angle_face_orientation__face_yaw_mean", EMOTION_PRESENCE, "Face Yaw", 412, hypothesis="H4", subcategory="Gesichtsorientierung", main=True, aliases=("face_yaw_mean",)),
    R("angle_face_orientation__angle_face_orientation", EMOTION_PRESENCE, "Kategoriale Gesichtsorientierung", 413, hypothesis="H4", subcategory="Gesichtsorientierung", main=True, kind="categorical"),
    R("angle_face_orientation__detected_face_frames", EMOTION_PRESENCE, "Anzahl erkannter Gesichtsframes", 414, hypothesis="H4", subcategory="Gesichtsorientierung", technical=True),
    R("structural_personen_anzahl__sampled_frames_personen_anzahl", EMOTION_PRESENCE, "Gesampelte Frames der Personenanalyse", 455, hypothesis="H4", subcategory="Personenanzahl", technical=True),
    R("caption_length_chars", TEXTUAL, "Caption-Länge Zeichen", 501, hypothesis="H5", main=True),
    R("caption_length_words", TEXTUAL, "Caption-Länge Wörter", 502, hypothesis="H5", main=True),
    R("caption_sentiment_index", TEXTUAL, "Caption-Sentiment-Index", 503, hypothesis="H5", main=True),
    R("comment_sentiment_index_mean", TEXTUAL, "Kommentar-Sentiment-Index", 504, hypothesis="H5", main=True, aliases=("sentiment_index_mean",)),
    R("comment_emotion_unique_labels_mean", TEXTUAL, "Kommentar-Emotionsvielfalt", 505, hypothesis="H5", main=True, aliases=("emotion_unique_labels_mean",)),
    R("topic_unique_labels", TEXTUAL, "Topic-Vielfalt", 506, hypothesis="H5", main=True),
    R("dominant_comment_emotion", TEXTUAL, "Dominante Kommentar-Emotion", 507, hypothesis="H5", main=True, kind="categorical"),
    R("dominant_topic_label", TEXTUAL, "Dominantes Kommentar-Topic", 508, hypothesis="H5", main=True, kind="categorical"),
    R("analysed_comment_count", TEXTUAL, "Anzahl analysierter Kommentare", 550, hypothesis="H5", technical=True),
    R("sentiment_index_median", TEXTUAL, "Median des Kommentar-Sentiment-Index", 551, hypothesis="H5", technical=True),
    R("comment_emotion_unique_labels_median", TEXTUAL, "Median der Kommentar-Emotionsvielfalt", 552, hypothesis="H5", technical=True, aliases=("emotion_unique_labels_median",)),
    R("caption_sentiment_negative", TEXTUAL, "Caption-Sentiment-Anteil negativ", 553, hypothesis="H5", technical=True),
    R("caption_sentiment_neutral", TEXTUAL, "Caption-Sentiment-Anteil neutral", 554, hypothesis="H5", technical=True),
    R("caption_sentiment_positive", TEXTUAL, "Caption-Sentiment-Anteil positiv", 555, hypothesis="H5", technical=True),
]

MODEL_CATEGORY_PREFIXES = {
    "brightness_contrast__": IMAGE_QUALITY,
    "saturation__": IMAGE_QUALITY,
    "visual_sharpness__": IMAGE_QUALITY,
    "cuts__": STRUCTURAL,
    "camera_distance__": STRUCTURAL,
    "scene_classification__": STRUCTURAL,
    "camera_stability__": STRUCTURAL,
    "skin_smoothness__": AESTHETICS,
    "aesthetic_quality__": AESTHETICS,
    "beauty_scoring__": AESTHETICS,
    "visual_filter__": AESTHETICS,
    "angle_face_orientation__": EMOTION_PRESENCE,
    "face_emotion__": EMOTION_PRESENCE,
    "body_pose__": EMOTION_PRESENCE,
    "structural_personen_anzahl__": EMOTION_PRESENCE,
    "topic_share__": TEXTUAL,
}

FACE_EMOTION_LABELS_DE = {
    "Angry": "Wut",
    "Disgust": "Ekel",
    "Fear": "Angst",
    "Happy": "Freude",
    "Neutral": "Neutral",
    "Sad": "Traurigkeit",
    "Surprise": "Überraschung",
}

POSE_LABELS_DE = {
    "calling": "Telefonieren",
    "clapping": "Klatschen",
    "cycling": "Radfahren",
    "dancing": "Tanzen",
    "drinking": "Trinken",
    "eating": "Essen",
    "fighting": "Kämpfen",
    "hugging": "Umarmen",
    "laughing": "Lachen",
    "listening_to_music": "Musik hören",
    "running": "Laufen",
    "sitting": "Sitzen",
    "sleeping": "Schlafen",
    "texting": "Textnachrichten schreiben",
    "using_laptop": "Laptop verwenden",
}

H4_SELECTED_DETECTION_MEASURES = {
    "face_emotion__detected_emotion_frames",
    "body_pose__detected_pose_frames",
    "structural_personen_anzahl__detected_person_frames",
    "angle_face_orientation__detected_face_frames",
}

H4_OVERVIEW_LABELS = {
    "Emotionsvielfalt": "Vielfalt erkannter Emotionen",
    "Personenanzahl": "Durchschnittliche Personenanzahl",
    "Face Pitch": "Mittlere vertikale Gesichtsausrichtung",
    "Face Yaw": "Mittlere horizontale Gesichtsausrichtung",
}

H4_CATEGORICAL_PLOT_FILES = {
    "face_emotion__emotion_major_beit_readable": "face_emotion_distribution",
    "body_pose__pose_orientation": "pose_distribution",
    "angle_face_orientation__angle_face_orientation": "face_orientation_distribution",
}


class MissingReport:
    def __init__(self) -> None:
        self.rows: list[tuple[str, str, str]] = []

    def add(self, planned: str, missing: str, consequence: str) -> None:
        self.rows.append((planned, missing, consequence))
        warnings.warn(f"{planned}: {missing} -> {consequence}", stacklevel=2)

    def write(self, path: Path) -> None:
        lines = [
            "# Fehlende Variablen",
            "",
            "| Geplante Darstellung | Fehlende Variable | Konsequenz |",
            "|---|---|---|",
        ]
        if self.rows:
            for planned, missing, consequence in self.rows:
                lines.append(f"| {planned} | {missing} | {consequence} |")
        else:
            lines.append("| Keine | Keine | Alle geplanten Pflichtausgaben mit verfügbaren Daten wurden erstellt |")
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def find_project_root() -> Path:
    here = Path(__file__).resolve()
    for candidate in [here.parent, *here.parents]:
        if (candidate / "data").exists() and (candidate / "comments").exists():
            return candidate
    return Path.cwd()


def setup_dirs(root: Path) -> dict[str, Path]:
    dirs = {
        "fig_main": root / "exports" / "report" / "figures",
        "fig_appendix": root / "exports" / "details" / "all_variables" / "figures",
        "tab_main": root / "exports" / "report" / "tables",
        "tab_appendix": root / "exports" / "details" / "additional_tables",
        "results_main_tables": root / "exports" / "details" / "selected_variables" / "tables",
        "results_appendix_tables": root / "exports" / "details" / "all_variables" / "tables",
        "results_main_plots": root / "exports" / "details" / "selected_variables" / "figures",
        "results_appendix_plots": root / "exports" / "details" / "all_variables" / "figures",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def rel(path: Path, root: Path) -> str:
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)


def read_csv_if_exists(path: Path, missing: MissingReport, label: str) -> pd.DataFrame | None:
    if not path.exists():
        missing.add(label, str(path), "Datei wurde nicht gefunden")
        return None
    return pd.read_csv(path)


def choose_master_file(root: Path) -> Path:
    candidates = [
        root / "data/04_analysis_results/visual_features/99_AI_AND_REAL_TIKTOK_VIDEOS_all_results_merged.csv",
        root / "data/04_analysis_results/visual_features/all_results_merged.csv",
        root / "data/04_analysis_results/visual_features/merged_results.csv",
        root / "data/04_analysis_results/visual_features/statistics_results.csv",
    ]
    for path in candidates:
        if path.exists():
            return path

    csvs = sorted(root.glob("**/*.csv"))
    scored: list[tuple[int, Path]] = []
    for path in csvs:
        lower = path.name.lower()
        score = 0
        for token in ["merged", "all_results", "statistics", "results"]:
            if token in lower:
                score += 10
        if "video" in lower:
            score += 5
        if "comment" in lower:
            score -= 4
        scored.append((score, path))
    if not scored:
        raise FileNotFoundError("Keine CSV-Datei im Projekt gefunden.")
    return sorted(scored, key=lambda item: (-item[0], len(str(item[1]))))[0][1]


def normalize_video_id(df: pd.DataFrame) -> pd.DataFrame:
    if "video_id" in df.columns:
        df["video_id_str"] = df["video_id"].astype("string").str.replace(r"\.0$", "", regex=True)
    elif "video_thread_id" in df.columns:
        df["video_id_str"] = df["video_thread_id"].astype("string").str.replace(r"\.0$", "", regex=True)
    return df


def normalize_group_value(value: object) -> str | float:
    if pd.isna(value):
        return np.nan
    cleaned = str(value).strip().lower()
    vi_values = {"ai", "ki", "vi", "virtual", "virtuell", "synthetic", "künstlich", "kuenstlich"}
    ri_values = {"real", "ri", "human", "menschlich", "mensch", "real influencer", "real_influencer"}
    if cleaned in vi_values or cleaned.startswith("ai"):
        return "VI"
    if cleaned in ri_values or cleaned.startswith("real"):
        return "RI"
    return np.nan


def add_group_column(df: pd.DataFrame, missing: MissingReport) -> pd.DataFrame:
    candidates = ["influencer_type", "Typ", "group", "type", "influencer"]
    for col in candidates:
        if col in df.columns:
            mapped = df[col].map(normalize_group_value)
            if mapped.notna().sum() > 0:
                df["Influencerinnen-Typ"] = pd.Categorical(mapped, categories=GROUP_ORDER, ordered=True)
                return df
    missing.add("Gruppierungsvariable", ", ".join(candidates), "Gruppenvergleiche und Gruppenplots können unvollständig sein")
    df["Influencerinnen-Typ"] = pd.Categorical([np.nan] * len(df), categories=GROUP_ORDER, ordered=True)
    return df


def first_existing(df: pd.DataFrame, aliases: Iterable[str]) -> str | None:
    for alias in aliases:
        if alias in df.columns:
            return alias
    lower_map = {c.lower(): c for c in df.columns}
    for alias in aliases:
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]
    return None


def add_derived_columns(df: pd.DataFrame, missing: MissingReport) -> pd.DataFrame:
    engagement = first_existing(df, ["video_engagement_rate", "engagement_rate", "engagement"])
    if engagement:
        df["engagement_rate"] = pd.to_numeric(df[engagement], errors="coerce")
    else:
        likes = first_existing(df, ["video_like_count", "like_count", "likes"])
        comments = first_existing(df, ["video_comment_count", "comment_count", "comments"])
        shares = first_existing(df, ["video_share_count", "share_count", "shares"])
        saves = first_existing(df, ["video_save_count", "save_count", "saves"])
        views = first_existing(df, ["video_view_count", "view_count", "views"])
        required = [likes, comments, shares, views]
        if all(required):
            numerator = pd.to_numeric(df[likes], errors="coerce")
            numerator += pd.to_numeric(df[comments], errors="coerce")
            numerator += pd.to_numeric(df[shares], errors="coerce")
            if saves:
                numerator += pd.to_numeric(df[saves], errors="coerce")
            denominator = pd.to_numeric(df[views], errors="coerce").replace(0, np.nan)
            df["engagement_rate"] = numerator / denominator
        else:
            missing.add(
                "Engagement-Rate",
                "video_engagement_rate oder Likes/Kommentare/Shares/Views",
                "Engagement-Analysen wurden übersprungen",
            )
    if "engagement_rate" in df.columns:
        df["log_engagement_rate"] = np.log1p(pd.to_numeric(df["engagement_rate"], errors="coerce"))

    if "seconds_per_cut" not in df.columns:
        duration_col = first_existing(df, ["video_duration_seconds", "duration_seconds", "cuts__video_duration_seconds_est"])
        cut_col = first_existing(df, ["cut_count", "cuts__cut_count"])
        if duration_col and cut_col:
            cuts = pd.to_numeric(df[cut_col], errors="coerce")
            duration = pd.to_numeric(df[duration_col], errors="coerce")
            df["seconds_per_cut"] = np.where(cuts > 0, duration / cuts, np.nan)
        else:
            missing.add("Sekunden pro Cut", "Videolänge oder Cut Count", "Feature wurde nicht berechnet")
    return df


def merge_text_results(root: Path, df: pd.DataFrame, missing: MissingReport) -> pd.DataFrame:
    sources = [
        (
            root / "comments/results/01_caption_sentiment_results.csv",
            "Caption-Sentiment",
            [
                "video_id",
                "caption_length_chars",
                "caption_length_words",
                "caption_sentiment_label",
                "caption_sentiment_index",
                "caption_sentiment_negative",
                "caption_sentiment_neutral",
                "caption_sentiment_positive",
            ],
            {},
        ),
        (
            root / "comments/results/02_comment_sentiment_video_level.csv",
            "Kommentar-Sentiment",
            ["video_id", "comment_count", "sentiment_index_mean", "sentiment_index_median", "dominant_comment_sentiment"],
            {"comment_count": "analysed_comment_count", "sentiment_index_mean": "comment_sentiment_index_mean"},
        ),
        (
            root / "comments/results/03_comment_emotion_video_level.csv",
            "Kommentar-Emotion",
            ["video_id", "dominant_comment_emotion", "emotion_unique_labels_mean", "emotion_unique_labels_median"],
            {"emotion_unique_labels_mean": "comment_emotion_unique_labels_mean"},
        ),
        (
            root / "comments/results/04_comment_topics_video_level.csv",
            "Kommentar-Topics",
            ["video_id", "topic_unique_labels", "dominant_topic_id", "dominant_topic_label"],
            {},
        ),
    ]
    df = normalize_video_id(df)
    if "video_id_str" not in df.columns:
        missing.add("Textuelle Zusatzdaten", "video_id", "Comment-Ergebnisse konnten nicht gemergt werden")
        return df

    for path, label, cols, rename_map in sources:
        if not path.exists():
            missing.add(label, str(path), "Zusatzdaten wurden nicht gemergt")
            continue
        extra = pd.read_csv(path)
        extra = normalize_video_id(extra)
        if "video_id_str" not in extra.columns:
            missing.add(label, "video_id", "Zusatzdaten wurden nicht gemergt")
            continue
        keep = [c for c in cols if c in extra.columns]
        if "video_id" in keep:
            keep.remove("video_id")
        if not keep:
            missing.add(label, ", ".join(cols), "Keine nutzbaren Spalten in Zusatzdatei")
            continue
        extra = extra[["video_id_str", *keep]].drop_duplicates("video_id_str")
        extra = extra.rename(columns=rename_map)
        for col in list(extra.columns):
            if col != "video_id_str" and col in df.columns:
                extra = extra.drop(columns=[col])
        df = df.merge(extra, on="video_id_str", how="left")

    topic_path = root / "comments/results/04_comment_topics_results.csv"
    if topic_path.exists():
        topics = normalize_video_id(pd.read_csv(topic_path))
        if {"video_id_str", "topic_id"}.issubset(topics.columns):
            topic_shares = pd.crosstab(topics["video_id_str"], topics["topic_id"], normalize="index")
            topic_shares.columns = [f"topic_share__{topic_id}" for topic_id in topic_shares.columns]
            df = df.merge(topic_shares.reset_index(), on="video_id_str", how="left")
        else:
            missing.add("Topic-Anteile", "video_id und/oder topic_id", "Einzelne Topic-Anteile wurden nicht gemergt")
    else:
        missing.add("Topic-Anteile", str(topic_path), "Einzelne Topic-Anteile wurden nicht gemergt")
    return df


def configure_style() -> None:
    sns.set_theme(
        context="paper",
        style="whitegrid",
        rc={
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 9,
            "figure.dpi": 120,
            "savefig.dpi": 300,
            "axes.spines.top": False,
            "axes.spines.right": False,
        },
    )


def save_figure(fig: plt.Figure, base_path: Path, created: list[Path]) -> None:
    out = base_path.with_suffix(".png")
    fig.savefig(out, bbox_inches="tight", dpi=300)
    created.append(out)
    plt.close(fig)


def export_table(df: pd.DataFrame, base_path: Path, created: list[Path]) -> None:
    csv_path = base_path.with_suffix(".csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    created.append(csv_path)


def export_reporting_table(
    df: pd.DataFrame,
    base_path: Path,
    created: list[Path],
    italicize_technical_rows: bool = False,
) -> None:
    """Export thesis-facing tables as CSV."""
    export_table(df, base_path, created)


def descriptive_stats(series: pd.Series) -> dict[str, float | int]:
    s = pd.to_numeric(series, errors="coerce").dropna()
    q1 = s.quantile(0.25) if len(s) else np.nan
    q3 = s.quantile(0.75) if len(s) else np.nan
    return {
        "n": int(s.count()),
        "M": s.mean() if len(s) else np.nan,
        "SD": s.std(ddof=1) if len(s) > 1 else np.nan,
        "Md": s.median() if len(s) else np.nan,
        "IQR": q3 - q1 if len(s) else np.nan,
        "Min": s.min() if len(s) else np.nan,
        "Max": s.max() if len(s) else np.nan,
    }


def direction_from_groups(df: pd.DataFrame, col: str) -> str:
    means = (
        df.groupby("Influencerinnen-Typ", observed=True)[col]
        .mean(numeric_only=True)
        .reindex(GROUP_ORDER)
    )
    if means.isna().any():
        return "nicht bestimmbar"
    if math.isclose(float(means["VI"]), float(means["RI"]), rel_tol=1e-9, abs_tol=1e-12):
        return "kein Mittelwertunterschied"
    return "VI höher" if means["VI"] > means["RI"] else "RI höher"


def round_numeric(df: pd.DataFrame, digits: int = 3) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].round(digits)
    return out


def round_reporting_table(df: pd.DataFrame, digits: int = 3) -> pd.DataFrame:
    """Round descriptive values while preserving exact p-values."""
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]) and not col.lower().startswith("p"):
            out[col] = out[col].round(digits)
    return out


def format_p_value(value: object) -> str:
    """Format p-values for thesis tables without reporting 0.000."""
    if pd.isna(value):
        return ""
    p = float(value)
    if p < 0.001:
        return "< .001"
    return f"{p:.3f}".lstrip("0")


def format_p_clause(value: object) -> str:
    formatted = format_p_value(value)
    return f"p {formatted}" if formatted.startswith("<") else f"p = {formatted}"


def format_p_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if col.lower() in {"p", "p-wert", "p_wert"}:
            out[col] = out[col].map(format_p_value)
    return out


def feature_column_map(df: pd.DataFrame, specs: Iterable[FeatureSpec]) -> dict[str, str]:
    mapping = {}
    for spec in specs:
        col = first_existing(df, spec.aliases)
        if col:
            mapping[spec.label] = col
    return mapping


def create_table_1(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]]) -> None:
    rows = []
    all_specs = [
        FeatureSpec(
            "Engagement-Rate",
            "Engagement",
            ("engagement_rate",),
            "Aus TikTok-Engagementmetriken",
            "(Likes + Kommentare + Shares ggf. Saves) / Views oder vorhandene Rate",
            "Höhere Werte stehen für relativ stärkeres Engagement.",
        ),
        *VISUAL_FEATURES,
        *TEXT_FEATURES,
    ]
    for spec in all_specs:
        col = first_existing(df, spec.aliases)
        if col:
            rows.append(
                {
                    "Kategorie": spec.category,
                    "Merkmal": spec.label,
                    "Variable im Datensatz": col,
                    "Methode/Modell": spec.method,
                    "Aggregation auf Videoebene": spec.aggregation,
                    "Interpretation": spec.interpretation,
                }
            )
    table = pd.DataFrame(rows)
    export_table(table, dirs["tab_main"] / "tab_01_operationalisierung_merkmale", created)
    overview.append(
        {
            "Kapitel": "4 Methodik",
            "Typ": "Tabelle 1",
            "Titel": "Operationalisierung der visuellen, textuellen und engagementbezogenen Merkmale",
            "Datei": "exports/report/tables/tab_01_operationalisierung_merkmale.csv",
            "Verwendung im Text": "Nach Beschreibung der Analysepipeline einfügen",
        }
    )


def create_table_2(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    rows = []
    for spec in VISUAL_FEATURES:
        col = first_existing(df, spec.aliases)
        if not col:
            missing.add("Tabelle 2", spec.label, "Feature wurde nicht aufgenommen")
            continue
        row = {"Feature": spec.label}
        for group in GROUP_ORDER:
            stats_dict = descriptive_stats(df.loc[df["Influencerinnen-Typ"] == group, col])
            for key, value in stats_dict.items():
                row[f"{group}: {key}"] = value
        row["Richtung"] = direction_from_groups(df, col)
        rows.append(row)
    if not rows:
        missing.add("Tabelle 2", "alle visuellen Features", "Tabelle wurde nicht erstellt")
        return
    table = round_numeric(pd.DataFrame(rows))
    export_table(table, dirs["tab_main"] / "tab_02_deskriptive_visuelle_merkmale", created)
    overview.append(
        {
            "Kapitel": "5.2 Visuelle Merkmale",
            "Typ": "Tabelle 2",
            "Titel": "Deskriptive Kennwerte zentraler visueller Merkmale nach Influencerinnen-Typ",
            "Datei": "exports/report/tables/tab_02_deskriptive_visuelle_merkmale.csv",
            "Verwendung im Text": "Vor Unterabschnitten 5.2.1-5.2.4 einfügen",
        }
    )


def cohen_d(x: pd.Series, y: pd.Series) -> float:
    """Calculate Cohen's d as RI minus VI when called with (VI, RI)."""
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()
    if len(x) < 2 or len(y) < 2:
        return np.nan
    pooled = math.sqrt(((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1)) / (len(x) + len(y) - 2))
    if pooled == 0:
        return np.nan
    return (y.mean() - x.mean()) / pooled


def create_table_3(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    rows = []
    for spec in [*VISUAL_FEATURES, *TEXT_FEATURES]:
        col = first_existing(df, spec.aliases)
        if not col:
            continue
        vi = pd.to_numeric(df.loc[df["Influencerinnen-Typ"] == "VI", col], errors="coerce").dropna()
        ri = pd.to_numeric(df.loc[df["Influencerinnen-Typ"] == "RI", col], errors="coerce").dropna()
        if len(vi) < 5 or len(ri) < 5:
            rows.append(
                {
                    "Kategorie": spec.category,
                    "Feature": spec.label,
                    "Test": "Mann-Whitney-U-Test; Welch-t-Test",
                    "Teststatistik": np.nan,
                    "p": np.nan,
                    "Effektgröße": np.nan,
                    "Richtung": "nicht bestimmbar",
                    "Status": "explorativ",
                }
            )
            continue
        try:
            mw = stats.mannwhitneyu(vi, ri, alternative="two-sided")
            rbc = (2 * mw.statistic / (len(vi) * len(ri))) - 1
        except Exception:
            mw = None
            rbc = np.nan
        try:
            tt = stats.ttest_ind(vi, ri, equal_var=False, nan_policy="omit")
        except Exception:
            tt = None

        p_mw = mw.pvalue if mw is not None else np.nan
        status = "signifikant" if pd.notna(p_mw) and p_mw < 0.05 else "nicht signifikant"
        rows.append(
            {
                "Kategorie": spec.category,
                "Feature": spec.label,
                "Test": "Mann-Whitney-U-Test",
                "Teststatistik": mw.statistic if mw is not None else np.nan,
                "p": p_mw,
                "Effektgröße": f"Rank-Biserial r = {rbc:.3f}" if pd.notna(rbc) else np.nan,
                "Richtung": direction_from_groups(df, col),
                "Status": status if mw is not None else "nicht berechnet",
            }
        )
        p_t = tt.pvalue if tt is not None else np.nan
        status_t = "signifikant" if pd.notna(p_t) and p_t < 0.05 else "nicht signifikant"
        d = cohen_d(vi, ri)
        rows.append(
            {
                "Kategorie": spec.category,
                "Feature": spec.label,
                "Test": "Welch-t-Test",
                "Teststatistik": tt.statistic if tt is not None else np.nan,
                "p": p_t,
                "Effektgröße": f"Cohen's d = {d:.3f}" if pd.notna(d) else np.nan,
                "Richtung": direction_from_groups(df, col),
                "Status": status_t if tt is not None else "nicht berechnet",
            }
        )
    if not rows:
        missing.add("Tabelle 3", "metrische visuelle/textuelle Features", "Tabelle wurde nicht erstellt")
        return
    table = format_p_columns(round_numeric(pd.DataFrame(rows)))
    export_table(table, dirs["tab_main"] / "tab_03_gruppenvergleiche_vi_ri", created)
    overview.append(
        {
            "Kapitel": "5.2/5.3 Ergebnisse",
            "Typ": "Tabelle 3",
            "Titel": "Gruppenvergleiche visueller und textueller Merkmale zwischen VI- und RI-Videos",
            "Datei": "exports/report/tables/tab_03_gruppenvergleiche_vi_ri.csv",
            "Verwendung im Text": "Bei Gruppenunterschieden und Signifikanztests referenzieren",
        }
    )


def create_table_4(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    rows = []
    for category, aliases in LABEL_FEATURES:
        col = first_existing(df, aliases)
        if not col:
            missing.add("Tabelle 4", category, "Labelverteilung wurde nicht aufgenommen")
            continue
        temp = df[["Influencerinnen-Typ", col]].dropna()
        labels = sorted(temp[col].astype(str).unique())
        denominators = temp.groupby("Influencerinnen-Typ", observed=True)[col].count().reindex(GROUP_ORDER).fillna(0)
        for label in labels:
            row = {"Kategorie": category, "Label": label}
            percentages = {}
            for group in GROUP_ORDER:
                n = int(((temp["Influencerinnen-Typ"] == group) & (temp[col].astype(str) == label)).sum())
                pct = (n / denominators[group] * 100) if denominators[group] else 0
                row[f"{group}: n"] = n
                row[f"{group}: %"] = pct
                percentages[group] = pct
            row["Differenz in Prozentpunkten"] = percentages["VI"] - percentages["RI"]
            rows.append(row)
    if not rows:
        missing.add("Tabelle 4", "textuelle Labelspalten", "Tabelle wurde nicht erstellt")
        return
    table = round_numeric(pd.DataFrame(rows))
    export_table(table, dirs["tab_main"] / "tab_04_textuelle_labels", created)
    overview.append(
        {
            "Kapitel": "5.3 Textuelle Merkmale",
            "Typ": "Tabelle 4",
            "Titel": "Verteilung textueller Labels nach Influencerinnen-Typ",
            "Datei": "exports/report/tables/tab_04_textuelle_labels.csv",
            "Verwendung im Text": "Sentiment-, Emotions- und Topic-Verteilungen berichten",
        }
    )


def correlation_strength(rho: float) -> str:
    abs_rho = abs(rho)
    if abs_rho < 0.10:
        return "vernachlässigbar"
    if abs_rho < 0.30:
        return "schwach"
    if abs_rho < 0.50:
        return "moderat"
    return "stark"


def correlation_interpretation(rho: float, p: float) -> str:
    if pd.isna(rho) or pd.isna(p):
        return "nicht berechnet"
    if p >= 0.05:
        return "kein statistisch signifikanter Zusammenhang im vorliegenden Sample"
    direction = "positiver" if rho > 0 else "negativer"
    return f"{correlation_strength(rho)}er {direction} Zusammenhang"


def create_table_5(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> pd.DataFrame:
    if "engagement_rate" not in df.columns:
        missing.add("Tabelle 5", "engagement_rate", "Tabelle wurde nicht erstellt")
        return pd.DataFrame()
    rows = []
    for spec in [*VISUAL_FEATURES, *TEXT_FEATURES]:
        col = first_existing(df, spec.aliases)
        if not col or col == "engagement_rate":
            continue
        temp = df[[col, "engagement_rate"]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(temp) < 5 or temp[col].nunique() < 2 or temp["engagement_rate"].nunique() < 2:
            rows.append(
                {
                    "Kategorie": spec.category,
                    "Feature": spec.label,
                    "Spearman ρ": np.nan,
                    "p": np.nan,
                    "Stärke": "nicht berechnet",
                    "Richtung": "nicht berechnet",
                    "Interpretation": "nicht berechnet",
                }
            )
            continue
        rho, p = stats.spearmanr(temp[col], temp["engagement_rate"])
        rows.append(
            {
                "Kategorie": spec.category,
                "Feature": spec.label,
                "Spearman ρ": rho,
                "p": p,
                "Stärke": correlation_strength(rho),
                "Richtung": "positiv" if rho > 0 else "negativ" if rho < 0 else "kein Zusammenhang",
                "Interpretation": correlation_interpretation(rho, p),
            }
        )
    if not rows:
        missing.add("Tabelle 5", "numerische Features und engagement_rate", "Tabelle wurde nicht erstellt")
        return pd.DataFrame()
    table = format_p_columns(round_numeric(pd.DataFrame(rows)))
    export_table(table, dirs["tab_main"] / "tab_05_engagement_korrelationen", created)
    overview.append(
        {
            "Kapitel": "5.4 Engagement",
            "Typ": "Tabelle 5",
            "Titel": "Zusammenhang zentraler Merkmale mit der Engagement-Rate",
            "Datei": "exports/report/tables/tab_05_engagement_korrelationen.csv",
            "Verwendung im Text": "Zentrale Korrelationen berichten",
        }
    )
    return table


MAIN_ENGAGEMENT_LABELS = {
    "Videolänge",
    "Szenenvielfalt",
    "Cut Count",
    "Cuts pro Sekunde",
    "Sekunden pro Cut",
    "Kameradistanzvariation",
    "Helligkeit",
    "Kontrast",
    "Sättigung",
    "Durchschnittliche Schärfe",
    "Streuung der Bildschärfe",
    "Aesthetic Score",
    "Beauty Score",
    "Beauty Score SD",
    "Hautglättung Highpass",
    "Hautglättung DoG",
    "Wahrscheinlichkeit für Filtereinsatz",
    "Face Pitch",
    "Face Yaw",
    "Pose Confidence",
    "Personenanzahl",
    "Emotionsvielfalt",
    "Caption-Länge Zeichen",
    "Caption-Länge Wörter",
    "Caption-Sentiment-Index",
    "Kommentar-Sentiment-Index",
    "Kommentar-Emotionsvielfalt",
    "Topic-Vielfalt",
}


def prettify_raw_name(raw_name: str) -> str:
    if raw_name.startswith("topic_share__"):
        return f"Topic-Anteil {raw_name.split('__', 1)[1]}"
    metric = raw_name.split("__", 1)[-1]
    return metric.replace("_", " ").strip().capitalize()


def category_for_raw_name(raw_name: str) -> str:
    for prefix, category in MODEL_CATEGORY_PREFIXES.items():
        if raw_name.startswith(prefix):
            return category
    return TEXTUAL if raw_name.startswith(("caption_", "comment_", "sentiment_", "topic_", "analysed_")) else "Weitere Variablen"


def looks_technical(raw_name: str) -> bool:
    return any(
        token in raw_name
        for token in (
            "has_frames",
            "has_video",
            "detected_",
            "processed_",
            "scored_frames",
            "sampled_",
            "frames_scanned",
            "video_fps",
            "cuts_per_frame",
            "confidence",
            "texture_",
            "selected_sigma",
        )
    )


def numeric_values(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors="coerce")


def reporting_specs_for_df(df: pd.DataFrame) -> list[ReportingFeatureSpec]:
    """Return configured specs plus newly calculated model metrics for the appendix."""
    specs = list(REPORTING_FEATURES)
    covered = {spec.raw_name for spec in specs}
    covered.update(alias for spec in specs for alias in spec.aliases)
    discovered_order = 900
    for raw_name in df.columns:
        if raw_name in covered or not any(raw_name.startswith(prefix) for prefix in MODEL_CATEGORY_PREFIXES):
            continue
        if numeric_values(df, raw_name).notna().sum() == 0:
            continue
        specs.append(
            R(
                raw_name,
                category_for_raw_name(raw_name),
                prettify_raw_name(raw_name),
                discovered_order,
                hypothesis="Zusatzmetrik",
                technical=looks_technical(raw_name),
            )
        )
        discovered_order += 1
    return specs


def resolve_reporting_specs(
    df: pd.DataFrame,
    specs: Iterable[ReportingFeatureSpec],
    *,
    main_only: bool = False,
    numeric_only: bool = False,
) -> list[tuple[ReportingFeatureSpec, str]]:
    resolved: list[tuple[ReportingFeatureSpec, str]] = []
    used_columns: set[str] = set()
    for spec in sorted(specs, key=lambda item: item.order):
        if main_only and not spec.include_main:
            continue
        aliases = (spec.raw_name, *spec.aliases)
        col = first_existing(df, aliases)
        if not col or col in used_columns:
            continue
        if numeric_only and (spec.kind != "numeric" or numeric_values(df, col).notna().sum() == 0):
            continue
        resolved.append((spec, col))
        used_columns.add(col)
    return resolved


def direction_from_d(value: float) -> str:
    if pd.isna(value):
        return "nicht bestimmbar"
    if math.isclose(float(value), 0.0, rel_tol=1e-9, abs_tol=1e-12):
        return "kein Mittelwertunterschied"
    return "RI höher" if value > 0 else "VI höher"


def calculate_group_comparisons(df: pd.DataFrame, specs: Iterable[tuple[ReportingFeatureSpec, str]]) -> pd.DataFrame:
    rows = []
    for spec, col in specs:
        vi = numeric_values(df.loc[df["Influencerinnen-Typ"] == "VI"], col).dropna()
        ri = numeric_values(df.loc[df["Influencerinnen-Typ"] == "RI"], col).dropna()
        if len(vi) < 2 or len(ri) < 2:
            continue
        try:
            mw = stats.mannwhitneyu(vi, ri, alternative="two-sided")
        except Exception:
            mw = None
        try:
            tt = stats.ttest_ind(vi, ri, equal_var=False, nan_policy="omit")
        except Exception:
            tt = None
        d = cohen_d(vi, ri)
        p_mw = mw.pvalue if mw is not None else np.nan
        rows.append(
            {
                "Hypothese": spec.hypothesis,
                "Kategorie": spec.category,
                "Unterkategorie": spec.subcategory,
                "Merkmal": spec.label,
                "Rohvariable": col,
                "Technische Variable": "ja" if spec.technical else "nein",
                "n VI": len(vi),
                "M VI": vi.mean(),
                "n RI": len(ri),
                "M RI": ri.mean(),
                "U": mw.statistic if mw is not None else np.nan,
                "p Mann-Whitney-U": p_mw,
                "t Welch": tt.statistic if tt is not None else np.nan,
                "p Welch": tt.pvalue if tt is not None else np.nan,
                "Cohen's d (RI - VI)": d,
                "Richtung": direction_from_d(d),
                "Signifikant p < .05": "*" if pd.notna(p_mw) and p_mw < 0.05 else "",
                "_order": spec.order,
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["_order", "Merkmal"]).drop(columns="_order")


def calculate_engagement_correlations(df: pd.DataFrame, specs: Iterable[tuple[ReportingFeatureSpec, str]]) -> pd.DataFrame:
    if "engagement_rate" not in df.columns:
        return pd.DataFrame()
    rows = []
    engagement = numeric_values(df, "engagement_rate")
    for spec, col in specs:
        temp = pd.DataFrame({"feature": numeric_values(df, col), "engagement": engagement}).dropna()
        if len(temp) < 5 or temp["feature"].nunique() < 2 or temp["engagement"].nunique() < 2:
            continue
        rho, p_value = stats.spearmanr(temp["feature"], temp["engagement"])
        rows.append(
            {
                "Hypothese": spec.hypothesis,
                "Kategorie": spec.category,
                "Merkmal": spec.label,
                "Rohvariable": col,
                "Technische Variable": "ja" if spec.technical else "nein",
                "n": len(temp),
                "Spearman rho": rho,
                "p": p_value,
                "Richtung": "positiv" if rho > 0 else "negativ" if rho < 0 else "kein Zusammenhang",
                "Signifikant p < .05": "*" if p_value < 0.05 else "",
                "_order": spec.order,
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["_order", "Merkmal"]).drop(columns="_order")


def plot_horizontal_values(
    table: pd.DataFrame,
    value_col: str,
    title: str,
    xlabel: str,
    base_path: Path,
    created: list[Path],
) -> None:
    if table.empty or value_col not in table.columns:
        return
    plot_df = table.dropna(subset=[value_col]).copy()
    if plot_df.empty:
        return
    category_order = {category: index for index, category in enumerate(CATEGORY_FILES)}
    plot_df["_category_order"] = plot_df["Kategorie"].map(category_order).fillna(999)
    plot_df["_abs"] = plot_df[value_col].abs()
    plot_df = plot_df.sort_values(["_category_order", "_abs"], ascending=[True, False])
    plot_df["Beschriftung"] = plot_df["Merkmal"].map(lambda value: wrapped(value, 36))
    colors = [COLOR_RI if value > 0 else COLOR_VI for value in plot_df[value_col]]
    fig, ax = plt.subplots(figsize=(9.5, max(5.0, 0.31 * len(plot_df))))
    ax.barh(plot_df["Beschriftung"], plot_df[value_col], color=colors)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Merkmal")
    ax.invert_yaxis()
    fig.tight_layout()
    save_figure(fig, base_path, created)


def feature_configuration_table(df: pd.DataFrame, specs: Iterable[ReportingFeatureSpec]) -> pd.DataFrame:
    rows = []
    for spec in sorted(specs, key=lambda item: item.order):
        actual_col = first_existing(df, (spec.raw_name, *spec.aliases))
        rows.append(
            {
                "Rohvariablenname": spec.raw_name,
                "Tatsächlich verwendete Spalte": actual_col or "",
                "Kategorie": spec.category,
                "Unterkategorie": spec.subcategory,
                "Merkmal": spec.label,
                "Hypothese": spec.hypothesis,
                "Haupttext": "ja" if spec.include_main else "nein",
                "Anhang": "ja" if spec.include_appendix else "nein",
                "Technisch/auswertungsbezogen": "ja" if spec.technical else "nein",
                "Reihenfolge": spec.order,
                "Datentyp": spec.kind,
                "Im Datensatz verfügbar": "ja" if actual_col else "nein",
            }
        )
    return pd.DataFrame(rows)


def create_label_distribution_table(
    df: pd.DataFrame,
    specs: Iterable[tuple[ReportingFeatureSpec, str]],
) -> pd.DataFrame:
    rows = []
    for spec, col in specs:
        if spec.kind != "categorical":
            continue
        temp = df[["Influencerinnen-Typ", col]].dropna()
        totals = temp.groupby("Influencerinnen-Typ", observed=True).size().reindex(GROUP_ORDER).fillna(0)
        for label in sorted(temp[col].astype(str).unique()):
            row = {
                "Kategorie": spec.category,
                "Merkmal": spec.label,
                "Rohvariable": col,
                "Ausprägung": categorical_display_value(spec, label),
            }
            for group in GROUP_ORDER:
                count = int(((temp["Influencerinnen-Typ"] == group) & (temp[col].astype(str) == label)).sum())
                row[f"{group}: n"] = count
                row[f"{group}: %"] = (count / totals[group] * 100) if totals[group] else 0
            rows.append(row)
    return round_numeric(pd.DataFrame(rows))


def categorical_display_value(spec: ReportingFeatureSpec, value: object) -> str:
    if spec.raw_name == "face_emotion__emotion_major_beit_readable":
        return FACE_EMOTION_LABELS_DE.get(str(value), str(value))
    if spec.raw_name == "body_pose__pose_orientation":
        return POSE_LABELS_DE.get(str(value), str(value))
    return str(value)


def benjamini_hochberg_adjust(p_values: pd.Series) -> pd.Series:
    """Return Benjamini-Hochberg adjusted p-values in the original row order."""
    values = pd.to_numeric(p_values, errors="coerce")
    valid = values.dropna().sort_values()
    adjusted = pd.Series(np.nan, index=values.index, dtype=float)
    if valid.empty:
        return adjusted
    running_min = 1.0
    for rank, index in reversed(list(enumerate(valid.index, start=1))):
        running_min = min(running_min, float(valid.loc[index]) * len(valid) / rank)
        adjusted.loc[index] = running_min
    return adjusted.clip(upper=1.0)


def calculate_categorical_posthoc_comparisons(
    df: pd.DataFrame,
    *,
    col: str,
    label_header: str,
    label_map: dict[str, str],
) -> pd.DataFrame:
    if col not in df.columns:
        return pd.DataFrame()
    temp = df[["Influencerinnen-Typ", col]].dropna()
    contingency = pd.crosstab(temp["Influencerinnen-Typ"], temp[col]).reindex(GROUP_ORDER).fillna(0).astype(int)
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return pd.DataFrame()
    totals = contingency.sum(axis=1)
    rows = []
    for label in contingency.columns:
        vi_n = int(contingency.loc["VI", label])
        ri_n = int(contingency.loc["RI", label])
        odds_ratio, p_value = stats.fisher_exact(
            [[vi_n, int(totals["VI"] - vi_n)], [ri_n, int(totals["RI"] - ri_n)]],
            alternative="two-sided",
        )
        rows.append(
            {
                label_header: label_map.get(str(label), str(label)),
                "Rohlabel": label,
                "VI: n": vi_n,
                "VI: %": vi_n / totals["VI"] * 100,
                "RI: n": ri_n,
                "RI: %": ri_n / totals["RI"] * 100,
                "Differenz VI - RI in Prozentpunkten": (vi_n / totals["VI"] - ri_n / totals["RI"]) * 100,
                "Odds Ratio VI vs. RI": odds_ratio,
                "p Fisher exakt": p_value,
            }
        )
    table = pd.DataFrame(rows)
    table["p FDR-BH"] = benjamini_hochberg_adjust(table["p Fisher exakt"])
    table["Explorativ p < .05"] = np.where(table["p Fisher exakt"] < 0.05, "*", "")
    table["FDR-korrigiert p < .05"] = np.where(table["p FDR-BH"] < 0.05, "*", "")
    return round_reporting_table(
        table.sort_values("Differenz VI - RI in Prozentpunkten", key=lambda values: values.abs(), ascending=False)
    )


def calculate_pose_posthoc_comparisons(df: pd.DataFrame) -> pd.DataFrame:
    return calculate_categorical_posthoc_comparisons(
        df,
        col="body_pose__pose_orientation",
        label_header="Pose",
        label_map=POSE_LABELS_DE,
    )


def calculate_face_emotion_posthoc_comparisons(df: pd.DataFrame) -> pd.DataFrame:
    return calculate_categorical_posthoc_comparisons(
        df,
        col="face_emotion__emotion_major_beit_readable",
        label_header="Gesichtsemotion",
        label_map={**FACE_EMOTION_LABELS_DE, "Neutral": "Neutralität"},
    )


def calculate_categorical_group_tests(
    df: pd.DataFrame,
    specs: Iterable[tuple[ReportingFeatureSpec, str]],
) -> pd.DataFrame:
    rows = []
    for spec, col in specs:
        if spec.kind != "categorical":
            continue
        temp = df[["Influencerinnen-Typ", col]].dropna()
        contingency = pd.crosstab(temp["Influencerinnen-Typ"], temp[col]).reindex(GROUP_ORDER).fillna(0)
        if contingency.shape[0] < 2 or contingency.shape[1] < 2:
            continue
        chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
        min_dim = min(contingency.shape) - 1
        cramer_v = math.sqrt(chi2 / (contingency.to_numpy().sum() * min_dim)) if min_dim else np.nan
        expected_lt_5 = int((expected < 5).sum())
        expected_total = int(expected.size)
        share_lt_5 = expected_lt_5 / expected_total if expected_total else np.nan
        if share_lt_5 > 0.20 or expected.min() < 1:
            note = "Chi-Quadrat-Voraussetzung eingeschränkt; seltene Kategorien zusammenfassen oder robust prüfen"
        elif expected_lt_5:
            note = "Einzelne seltene Kategorien; Chi-Quadrat-Ergebnis vorsichtig interpretieren"
        else:
            note = ""
        rows.append(
            {
                "Hypothese": spec.hypothesis,
                "Kategorie": spec.category,
                "Unterkategorie": spec.subcategory,
                "Merkmal": spec.label,
                "Rohvariable": col,
                "n": len(temp),
                "Chi-Quadrat": chi2,
                "df": dof,
                "p": p_value,
                "Cramér's V": cramer_v,
                "Erwartete Zellen < 5": expected_lt_5,
                "Anteil erwarteter Zellen < 5": share_lt_5,
                "Hinweis": note,
                "_order": spec.order,
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["_order", "Merkmal"]).drop(columns="_order")


def create_h4_overview_table(
    df: pd.DataFrame,
    specs: Iterable[tuple[ReportingFeatureSpec, str]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    selected = [
        (spec, col)
        for spec, col in specs
        if spec.category == EMOTION_PRESENCE and (spec.include_main or col in H4_SELECTED_DETECTION_MEASURES)
    ]
    full_table = calculate_group_comparisons(df, selected)
    if full_table.empty:
        return pd.DataFrame(), pd.DataFrame()
    full_table["Merkmal"] = full_table["Merkmal"].replace(H4_OVERVIEW_LABELS)
    full_table["Technisches Detektions-/Sichtbarkeitsmaß"] = full_table["Technische Variable"]
    full_table.loc[
        full_table["Technisches Detektions-/Sichtbarkeitsmaß"] == "ja",
        "Merkmal",
    ] += "*"
    compact_table = full_table[
        [
            "Unterkategorie",
            "Merkmal",
            "M VI",
            "M RI",
            "p Mann-Whitney-U",
            "Cohen's d (RI - VI)",
            "Richtung",
            "Technisches Detektions-/Sichtbarkeitsmaß",
        ]
    ].rename(
        columns={
            "Unterkategorie": "Kategorie",
            "M VI": "M (VI)",
            "M RI": "M (RI)",
            "p Mann-Whitney-U": "p",
        }
    )
    compact_table = round_reporting_table(compact_table)
    compact_table["p"] = compact_table["p"].map(format_p_value)
    return compact_table, round_reporting_table(full_table)


def plot_categorical_distribution(
    df: pd.DataFrame,
    spec: ReportingFeatureSpec,
    col: str,
    title: str,
    base_path: Path,
    created: list[Path],
) -> None:
    temp = df[["Influencerinnen-Typ", col]].dropna().copy()
    if temp.empty:
        return
    temp["Ausprägung"] = temp[col].map(lambda value: categorical_display_value(spec, value))
    plot_df = temp.groupby(["Influencerinnen-Typ", "Ausprägung"], observed=True).size().reset_index(name="n")
    plot_df["Anteil in %"] = plot_df["n"] / plot_df.groupby("Influencerinnen-Typ", observed=True)["n"].transform("sum") * 100
    order = plot_df.groupby("Ausprägung")["n"].sum().sort_values(ascending=False).index.tolist()
    fig, ax = plt.subplots(figsize=(max(8.5, 0.7 * len(order)), 5.2))
    sns.barplot(
        data=plot_df,
        x="Ausprägung",
        y="Anteil in %",
        hue="Influencerinnen-Typ",
        order=order,
        hue_order=GROUP_ORDER,
        palette=PALETTE,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel(spec.label)
    ax.set_ylabel("Anteil der Videos innerhalb der Gruppe in %")
    ax.tick_params(axis="x", rotation=22)
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles, ["VI", "RI"], title="Influencerinnen-Typ")
    fig.tight_layout()
    save_figure(fig, base_path, created)


def plot_h4_appendix_posthoc_distribution(
    table: pd.DataFrame,
    *,
    label_col: str,
    title: str,
    base_path: Path,
    created: list[Path],
    show_exploratory: bool,
) -> None:
    """Create readable appendix plots with percentages and post-hoc markers."""
    if table.empty:
        return
    plot_df = table.copy()
    plot_df["_max_share"] = plot_df[["VI: %", "RI: %"]].max(axis=1)
    plot_df = plot_df.sort_values("_max_share", ascending=True)
    y = np.arange(len(plot_df))
    height = 0.36
    max_share = float(plot_df["_max_share"].max())

    fig, ax = plt.subplots(figsize=(10.5, max(5.2, 0.46 * len(plot_df) + 1.6)))
    vi_bars = ax.barh(y - height / 2, plot_df["VI: %"], height=height, color=COLOR_VI, label="VI")
    ri_bars = ax.barh(y + height / 2, plot_df["RI: %"], height=height, color=COLOR_RI, label="RI")

    for bars in [vi_bars, ri_bars]:
        ax.bar_label(bars, fmt="%.1f %%", padding=3, fontsize=8.5)

    marker_x = max_share + max(7.0, max_share * 0.16)
    for yi, (_, row) in enumerate(plot_df.iterrows()):
        if row["FDR-korrigiert p < .05"] == "*":
            ax.text(marker_x, yi, "*", va="center", ha="center", fontsize=13, fontweight="bold")
        elif show_exploratory and row["Explorativ p < .05"] == "*":
            ax.text(marker_x, yi, "†", va="center", ha="center", fontsize=12, fontweight="bold")

    ax.set_yticks(y, plot_df[label_col])
    ax.set_xlim(0, marker_x + max(3.5, max_share * 0.07))
    ax.set_xlabel("Anteil der Videos innerhalb der Gruppe in %")
    ax.set_ylabel("")
    ax.set_title(title, pad=14)
    ax.legend(title="Influencerinnen-Typ", loc="lower right")
    note = "* FDR-korrigierter emotionsweiser Fisher-Test: p < .05"
    if show_exploratory:
        note = "† Explorativ: poseweiser Fisher-Test p < .05; nach FDR-Korrektur keine einzelne Pose signifikant"
    ax.text(0.0, -0.13, note, transform=ax.transAxes, ha="left", va="top", fontsize=8.8, color="#444444")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    save_figure(fig, base_path, created)


def write_h4_result_text(
    df: pd.DataFrame,
    dirs: dict[str, Path],
    created: list[Path],
    h4_table: pd.DataFrame,
    h4_categorical_tests: pd.DataFrame,
    pose_posthoc: pd.DataFrame,
    face_emotion_posthoc: pd.DataFrame,
) -> None:
    if h4_table.empty or h4_categorical_tests.empty or pose_posthoc.empty or face_emotion_posthoc.empty:
        return

    def metric(label: str) -> pd.Series:
        return h4_table.loc[h4_table["Merkmal"].str.rstrip("*") == label].iloc[0]

    def categorical(label: str) -> pd.Series:
        return h4_categorical_tests.loc[h4_categorical_tests["Merkmal"] == label].iloc[0]

    emotion_confidence = metric("Emotionssicherheit")
    emotion_diversity = metric("Vielfalt erkannter Emotionen")
    pose_confidence = metric("Pose Confidence")
    persons = metric("Durchschnittliche Personenanzahl")
    max_persons = metric("Maximale Personenanzahl")
    pitch = metric("Mittlere vertikale Gesichtsausrichtung")
    yaw = metric("Mittlere horizontale Gesichtsausrichtung")
    face_categories = categorical("Dominante Gesichtsemotion")
    pose_categories = categorical("Dominante Körperpose")
    orientation_categories = categorical("Kategoriale Gesichtsorientierung")

    emotion_col = "face_emotion__emotion_major_beit_readable"
    emotion_counts = pd.crosstab(df["Influencerinnen-Typ"], df[emotion_col]).reindex(GROUP_ORDER).fillna(0)
    emotion_shares = emotion_counts.div(emotion_counts.sum(axis=1), axis=0) * 100

    def emotion_share(label: str) -> str:
        text_label = "Neutralität" if label == "Neutral" else FACE_EMOTION_LABELS_DE[label]
        return (
            f"{text_label} "
            f"({emotion_shares.loc['VI', label]:.1f} % vs. {emotion_shares.loc['RI', label]:.1f} %)"
        )

    significant_emotions = face_emotion_posthoc.loc[face_emotion_posthoc["p FDR-BH"] < 0.05].copy()

    def emotion_significance_list(table: pd.DataFrame) -> str:
        entries = [
            f"{row['Gesichtsemotion']} ({row['VI: %']:.1f} % vs. {row['RI: %']:.1f} %, p_FDR = {format_p_value(row['p FDR-BH'])})"
            for _, row in table.iterrows()
        ]
        if len(entries) < 2:
            return "".join(entries)
        return ", ".join(entries[:-1]) + " und " + entries[-1]

    exploratory_poses = pose_posthoc.loc[pose_posthoc["p Fisher exakt"] < 0.05].copy()
    vi_poses = exploratory_poses.loc[exploratory_poses["Differenz VI - RI in Prozentpunkten"] > 0]
    ri_poses = exploratory_poses.loc[exploratory_poses["Differenz VI - RI in Prozentpunkten"] < 0]

    def pose_list(table: pd.DataFrame) -> str:
        entries = [
            f"{row['Pose']} ({row['VI: %']:.1f} % vs. {row['RI: %']:.1f} %)"
            for _, row in table.iterrows()
        ]
        if len(entries) < 2:
            return "".join(entries)
        return ", ".join(entries[:-1]) + " und " + entries[-1]

    corrected_pose_count = int((pose_posthoc["p FDR-BH"] < 0.05).sum())
    pose_correction_note = (
        "Nach FDR-Korrektur bleibt jedoch keine einzelne Pose signifikant."
        if corrected_pose_count == 0
        else f"Nach FDR-Korrektur bleiben {corrected_pose_count} Einzelvergleiche signifikant."
    )

    text = f"""# Ergebnisbaustein: Emotion und Präsenz

Die Kategorie Emotion und Präsenz umfasst Gesichtsemotionen, Körperpose, Personenanzahl und Gesichtsorientierung. Die mit einem Stern markierten Detektions- und Sichtbarkeitsmaße sind technische Kontextvariablen. Die vorhandenen Rohvariablen enthalten Frame-Anzahlen, keine normalisierten Anteile.

Im Bereich der Gesichtsemotionen zeigen sich mehrere systematische Unterschiede. Die Sicherheit der dominanten Emotion ist bei RI-Videos signifikant höher als bei VI-Videos (Mann-Whitney-U: U = {emotion_confidence['U']:.1f}, {format_p_clause(emotion_confidence['p Mann-Whitney-U'])}, Cohen's d = {emotion_confidence["Cohen's d (RI - VI)"]:.2f}). Ausgeprägter ist der Unterschied in der Vielfalt erkannter Emotionen: RI-Videos weisen eine höhere emotionale Variabilität auf (U = {emotion_diversity['U']:.1f}, {format_p_clause(emotion_diversity['p Mann-Whitney-U'])}, d = {emotion_diversity["Cohen's d (RI - VI)"]:.2f}). Auch die Verteilung der dominanten Gesichtsemotionen unterscheidet sich zwischen den Gruppen (Chi-Quadrat = {face_categories['Chi-Quadrat']:.2f}, df = {int(face_categories['df'])}, {format_p_clause(face_categories['p'])}, Cramér's V = {face_categories["Cramér's V"]:.2f}). VI-Videos zeigen insbesondere häufiger {emotion_share('Happy')} und {emotion_share('Surprise')}; RI-Videos häufiger {emotion_share('Neutral')}, {emotion_share('Sad')} und {emotion_share('Disgust')}. In emotionsweisen Fisher-Tests bleiben nach FDR-Korrektur {emotion_significance_list(significant_emotions)} signifikant häufiger bei VI. Die übrigen Einzelunterschiede sind nach FDR-Korrektur nicht signifikant. {face_categories['Hinweis']}.

Für die Körperpose weisen VI-Videos eine höhere Pose Confidence auf als RI-Videos (U = {pose_confidence['U']:.1f}, {format_p_clause(pose_confidence['p Mann-Whitney-U'])}, d = {pose_confidence["Cohen's d (RI - VI)"]:.2f}). Die Verteilung der dominanten Posen unterscheidet sich ebenfalls (Chi-Quadrat = {pose_categories['Chi-Quadrat']:.2f}, df = {int(pose_categories['df'])}, {format_p_clause(pose_categories['p'])}, Cramér's V = {pose_categories["Cramér's V"]:.2f}). Deskriptiv und in explorativen poseweisen Fisher-Tests treten bei VI-Videos häufiger {pose_list(vi_poses)} auf. Bei RI-Videos treten häufiger {pose_list(ri_poses)} auf. {pose_correction_note} Der kategoriale Pose-Befund ist wegen mehrerer seltener Kategorien und der multiplen Einzelvergleiche vorsichtig zu interpretieren.

VI-Videos enthalten im Durchschnitt mehr gleichzeitig sichtbare Personen als RI-Videos (U = {persons['U']:.1f}, {format_p_clause(persons['p Mann-Whitney-U'])}, d = {persons["Cohen's d (RI - VI)"]:.2f}). Die maximale Personenanzahl unterscheidet sich nicht signifikant (U = {max_persons['U']:.1f}, {format_p_clause(max_persons['p Mann-Whitney-U'])}).

Für die Gesichtsorientierung ergeben sich weder bei Face Pitch (U = {pitch['U']:.1f}, {format_p_clause(pitch['p Mann-Whitney-U'])}) noch bei Face Yaw (U = {yaw['U']:.1f}, {format_p_clause(yaw['p Mann-Whitney-U'])}) signifikante Unterschiede. Auch die kategoriale Verteilung der Gesichtsorientierung unterscheidet sich nicht signifikant (Chi-Quadrat = {orientation_categories['Chi-Quadrat']:.2f}, df = {int(orientation_categories['df'])}, {format_p_clause(orientation_categories['p'])}).

Zusammenfassend zeigen sich differenzierte Unterschiede in der emotionalen und visuellen Inszenierung. RI-Videos weisen eine größere Vielfalt erkannter Gesichtsemotionen und eine etwas höhere Emotionssicherheit auf. VI-Videos zeigen eine höhere Pose Confidence und eine höhere durchschnittliche Personenanzahl. Da lediglich visuelle Proxies gemessen wurden, wird nicht von sozialer Präsenz gesprochen.
"""
    path = dirs["results_main_tables"] / "main_h4_emotion_presence_result_text.md"
    path.write_text(text, encoding="utf-8")
    created.append(path)


def create_systematic_reporting_outputs(
    df: pd.DataFrame,
    dirs: dict[str, Path],
    created: list[Path],
    missing: MissingReport,
) -> None:
    specs = reporting_specs_for_df(df)
    main_numeric = resolve_reporting_specs(df, specs, main_only=True, numeric_only=True)
    appendix_numeric = resolve_reporting_specs(df, specs, numeric_only=True)
    main_all = resolve_reporting_specs(df, specs, main_only=True)
    appendix_all = resolve_reporting_specs(df, specs)

    config_table = feature_configuration_table(df, specs)
    export_reporting_table(config_table, dirs["results_appendix_tables"] / "appendix_feature_reporting_configuration", created)

    for category, (hypothesis, filename) in CATEGORY_FILES.items():
        label_specs = [(spec, col) for spec, col in main_all if spec.category == category and spec.kind == "categorical"]
        labels = create_label_distribution_table(df, label_specs)
        if not labels.empty:
            export_reporting_table(labels, dirs["results_main_tables"] / f"main_{hypothesis}_{filename}_label_distributions", created)
        categorical_tests = round_reporting_table(calculate_categorical_group_tests(df, label_specs))
        if not categorical_tests.empty:
            export_reporting_table(categorical_tests, dirs["results_main_tables"] / f"main_{hypothesis}_{filename}_categorical_group_tests", created)

    for category, (hypothesis, filename) in CATEGORY_FILES.items():
        selected_specs = [(spec, col) for spec, col in main_numeric if spec.category == category]
        appendix_specs = [(spec, col) for spec, col in appendix_numeric if spec.category == category]
        main_table = round_reporting_table(calculate_group_comparisons(df, selected_specs))
        appendix_table = round_reporting_table(calculate_group_comparisons(df, appendix_specs))
        if main_table.empty:
            missing.add(f"Haupttext-Gruppenvergleich {category}", "verfügbare numerische Haupttextmerkmale", "Tabelle und Plot wurden nicht erstellt")
        else:
            export_reporting_table(main_table, dirs["results_main_tables"] / f"main_{hypothesis}_{filename}_group_comparison", created)
            plot_horizontal_values(
                main_table,
                "Cohen's d (RI - VI)",
                f"{category}: Gruppenunterschiede",
                "Cohen's d (positiv = RI höher, negativ = VI höher)",
                dirs["results_main_plots"] / f"main_{hypothesis}_{filename}_group_comparison",
                created,
            )
        if not appendix_table.empty:
            export_reporting_table(appendix_table, dirs["results_appendix_tables"] / f"appendix_{hypothesis}_{filename}_all_variables", created)
            plot_horizontal_values(
                appendix_table,
                "Cohen's d (RI - VI)",
                f"{category}: vollständige Gruppenunterschiede",
                "Cohen's d (positiv = RI höher, negativ = VI höher)",
                dirs["results_appendix_plots"] / f"appendix_{hypothesis}_{filename}_all_variables",
                created,
            )

    h4_overview, h4_overview_full = create_h4_overview_table(df, appendix_numeric)
    h4_label_specs = [
        (spec, col)
        for spec, col in appendix_all
        if spec.category == EMOTION_PRESENCE and spec.kind == "categorical"
    ]
    h4_categorical_tests = round_reporting_table(calculate_categorical_group_tests(df, h4_label_specs))
    pose_posthoc = calculate_pose_posthoc_comparisons(df)
    face_emotion_posthoc = calculate_face_emotion_posthoc_comparisons(df)
    if not h4_overview.empty:
        export_reporting_table(
            h4_overview,
            dirs["results_main_tables"] / "main_h4_emotion_presence_overview_with_detection_measures",
            created,
            italicize_technical_rows=True,
        )
        plot_horizontal_values(
            h4_overview,
            "Cohen's d (RI - VI)",
            "Emotion und Präsenz: Effektgrößen einschließlich Detektionsmaße",
            "Cohen's d (positiv = RI höher, negativ = VI höher)",
            dirs["results_main_plots"] / "main_h4_emotion_presence_effect_sizes_with_detection_measures",
            created,
        )
    if not h4_categorical_tests.empty:
        export_reporting_table(
            h4_categorical_tests,
            dirs["results_main_tables"] / "main_h4_emotion_presence_categorical_group_tests",
            created,
        )
    if not pose_posthoc.empty:
        export_reporting_table(
            pose_posthoc,
            dirs["results_main_tables"] / "main_h4_pose_posthoc_comparisons",
            created,
        )
    if not face_emotion_posthoc.empty:
        export_reporting_table(
            face_emotion_posthoc,
            dirs["results_main_tables"] / "main_h4_face_emotion_posthoc_comparisons",
            created,
        )
        plot_h4_appendix_posthoc_distribution(
            face_emotion_posthoc,
            label_col="Gesichtsemotion",
            title="Dominante Gesichtsemotionen nach Influencerinnen-Typ",
            base_path=dirs["results_appendix_plots"] / "appendix_h4_face_emotion_group_distribution",
            created=created,
            show_exploratory=False,
        )
    if not pose_posthoc.empty:
        plot_h4_appendix_posthoc_distribution(
            pose_posthoc,
            label_col="Pose",
            title="Dominante Körperposen nach Influencerinnen-Typ",
            base_path=dirs["results_appendix_plots"] / "appendix_h4_pose_group_distribution",
            created=created,
            show_exploratory=True,
        )
    if not h4_categorical_tests.empty and not pose_posthoc.empty and not face_emotion_posthoc.empty:
        write_h4_result_text(
            df,
            dirs,
            created,
            h4_overview_full,
            h4_categorical_tests,
            pose_posthoc,
            face_emotion_posthoc,
        )
    for spec, col in h4_label_specs:
        plot_dir = dirs["results_main_plots"] if spec.raw_name == "face_emotion__emotion_major_beit_readable" else dirs["results_appendix_plots"]
        plot_categorical_distribution(
            df,
            spec,
            col,
            f"{spec.label} nach Influencerinnen-Typ",
            plot_dir / f"{'main' if plot_dir == dirs['results_main_plots'] else 'appendix'}_h4_{H4_CATEGORICAL_PLOT_FILES[spec.raw_name]}",
            created,
        )

    main_corr_specs = [(spec, col) for spec, col in main_numeric if spec.label in MAIN_ENGAGEMENT_LABELS]
    main_corr = round_reporting_table(calculate_engagement_correlations(df, main_corr_specs))
    appendix_corr = round_reporting_table(calculate_engagement_correlations(df, appendix_numeric))
    if main_corr.empty:
        missing.add("Zentrale Merkmale und Engagement", "engagement_rate oder Haupttextmerkmale", "Tabelle und Plot wurden nicht erstellt")
    else:
        export_reporting_table(main_corr, dirs["results_main_tables"] / "main_engagement_correlations_selected", created)
        plot_horizontal_values(
            main_corr,
            "Spearman rho",
            "Zentrale Merkmale und Engagement",
            "Spearman rho (positiv = positiver Zusammenhang, negativ = negativer Zusammenhang)",
            dirs["results_main_plots"] / "main_engagement_correlations_selected",
            created,
        )
    if not appendix_corr.empty:
        export_reporting_table(appendix_corr, dirs["results_appendix_tables"] / "appendix_engagement_correlations_all", created)
        plot_horizontal_values(
            appendix_corr,
            "Spearman rho",
            "Vollständige Korrelationsübersicht aller Merkmale mit Engagement",
            "Spearman rho",
            dirs["results_appendix_plots"] / "appendix_engagement_correlations_all",
            created,
        )


def human_interpretation(row: pd.Series) -> str:
    n = row.get("n_rated_items", row.get("n", 0))
    mean = row.get("mean_rating", row.get("M", np.nan))
    two_rate = row.get("two_annotator_rate", np.nan)
    if pd.isna(n) or n == 0:
        return "zu prüfen"
    if n >= 30 and pd.notna(mean) and mean >= 3.5 and (pd.isna(two_rate) or two_rate >= 0.80):
        return "hoch belastbar"
    if n >= 20 and pd.notna(mean) and mean >= 3.0:
        return "mittel belastbar"
    if n >= 5:
        return "explorativ"
    return "zu prüfen"


def load_human_summary(root: Path) -> pd.DataFrame | None:
    path = root / "data/04_analysis_results/human_evaluation/human_eval_consensus_summary_overall_feature.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if "is_overall" in df.columns:
        df = df[df["is_overall"] == False].copy()  # noqa: E712
    return df


def load_human_item_stats(root: Path) -> pd.DataFrame | None:
    path = root / "data/04_analysis_results/human_evaluation/human_eval_consensus_item_ratings.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if "is_overall" in df.columns:
        df = df[df["is_overall"] == False].copy()  # noqa: E712
    if "consensus_rating" not in df.columns:
        return None
    df["consensus_rating"] = pd.to_numeric(df["consensus_rating"], errors="coerce")
    df = df.dropna(subset=["consensus_rating"])
    if df.empty:
        return None

    rows = []
    for (eval_column, feature), group in df.groupby(["eval_column", "feature"], observed=True):
        ratings = group["consensus_rating"].dropna()
        q1 = ratings.quantile(0.25)
        q3 = ratings.quantile(0.75)
        rows.append(
            {
                "eval_column": eval_column,
                "feature": feature,
                "n_item": int(ratings.count()),
                "M_item": ratings.mean(),
                "SD_item": ratings.std(ddof=1),
                "Md_item": ratings.median(),
                "IQR_item": q3 - q1,
                "Min_item": ratings.min(),
                "Max_item": ratings.max(),
            }
        )
    return pd.DataFrame(rows)


def create_table_6(root: Path, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> pd.DataFrame:
    human = load_human_summary(root)
    if human is None or human.empty:
        missing.add("Tabelle 6", "Human-Evaluation-Daten", "Tabelle wurde nicht erstellt")
        return pd.DataFrame()
    item_stats = load_human_item_stats(root)
    if item_stats is not None and not item_stats.empty:
        human = human.merge(item_stats, on=["eval_column", "feature"], how="left")
    else:
        missing.add("Tabelle 6", "human_eval_consensus_item_ratings.csv", "IQR, Min und Max konnten nicht aus Einzelratings berechnet werden")
    rows = []
    for _, row in human.iterrows():
        rows.append(
            {
                "Kategorie": row.get("feature", row.get("feature_group", "unbekannt")),
                "n": row.get("n_item", row.get("n_rated_items", row.get("n_items", np.nan))),
                "M": row.get("M_item", row.get("mean_rating", np.nan)),
                "SD": row.get("SD_item", row.get("sd_rating", np.nan)),
                "Md": row.get("Md_item", row.get("median_rating", np.nan)),
                "IQR": row.get("IQR_item", np.nan),
                "Min": row.get("Min_item", np.nan),
                "Max": row.get("Max_item", np.nan),
                "Interpretationsklasse": human_interpretation(row),
            }
        )
    table = round_numeric(pd.DataFrame(rows))
    export_table(table, dirs["tab_main"] / "tab_06_human_evaluation", created)
    overview.append(
        {
            "Kapitel": "5.1 Modellvalidierung",
            "Typ": "Tabelle 6",
            "Titel": "Ergebnisse der Human Evaluation nach Merkmalskategorie",
            "Datei": "exports/report/tables/tab_06_human_evaluation.csv",
            "Verwendung im Text": "Zur Validierung der Modelloutputs referenzieren",
        }
    )
    return human


def create_table_7(dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]]) -> None:
    rows = [
        {
            "Hypothese": "H1a Dynamik",
            "Zentrales Ergebnis": "RI-Videos weisen signifikant mehr Schnitte und höhere Werte bei Cuts pro Sekunde auf; Sekunden pro Cut ist im Mann-Whitney-Test bei VI höher.",
            "Interpretation": "Die visuellen Dynamikindikatoren sprechen insgesamt für eine stärkere Schnittdynamik menschlicher Influencerinnen-Videos.",
            "Status": "gestützt",
            "Verweis auf Tabelle/Abbildung": "Tabelle 3; Abbildung 2",
        },
        {
            "Hypothese": "H1b Dynamik-Engagement",
            "Zentrales Ergebnis": "Cut Count, Cuts pro Sekunde und Sekunden pro Cut zeigen keinen statistisch signifikanten Spearman-Zusammenhang mit der Engagement-Rate.",
            "Interpretation": "Für einen belastbaren Zusammenhang zwischen Schnittdynamik und Engagement findet sich im vorliegenden Sample keine Evidenz.",
            "Status": "nicht gestützt",
            "Verweis auf Tabelle/Abbildung": "Tabelle 5; Abbildung 7",
        },
        {
            "Hypothese": "H2 Bildtechnik",
            "Zentrales Ergebnis": "Sättigung ist bei VI signifikant höher, Schärfe bei RI signifikant höher; Helligkeit ist nur im Welch-t-Test signifikant, Kontrast nicht signifikant.",
            "Interpretation": "Bildtechnische Unterschiede liegen vor, fallen aber je nach Merkmal unterschiedlich aus.",
            "Status": "teilweise gestützt",
            "Verweis auf Tabelle/Abbildung": "Tabelle 3; Abbildung 3",
        },
        {
            "Hypothese": "H3 Ästhetik",
            "Zentrales Ergebnis": "Der Aesthetic Score ist bei VI signifikant höher; Beauty Score ist nicht signifikant, Beauty Score SD nur im Mann-Whitney-Test signifikant.",
            "Interpretation": "Die automatisierte Ästhetikbewertung stützt Unterschiede zugunsten von VI, während die Schönheitsbewertung weniger eindeutig ist.",
            "Status": "teilweise gestützt",
            "Verweis auf Tabelle/Abbildung": "Tabelle 3; Abbildung 2",
        },
        {
            "Hypothese": "H4 Emotion und Präsenz",
            "Zentrales Ergebnis": "Pose Confidence und Personenanzahl sind bei VI signifikant höher; Face Pitch und Face Yaw unterscheiden sich nicht signifikant. Face Pitch korreliert schwach negativ mit Engagement.",
            "Interpretation": "Einzelne Präsenzindikatoren unterscheiden sich zwischen VI und RI, der Befund ist jedoch nicht über alle Merkmale konsistent.",
            "Status": "teilweise gestützt",
            "Verweis auf Tabelle/Abbildung": "Tabelle 3; Tabelle 5; Abbildung 7",
        },
        {
            "Hypothese": "H5 Caption",
            "Zentrales Ergebnis": "Caption-Länge und Caption-Sentiment unterscheiden sich nicht signifikant zwischen VI und RI; Caption-Wortzahl zeigt nur einen schwachen positiven Engagement-Zusammenhang.",
            "Interpretation": "Die untersuchten Caption-Merkmale liefern nur begrenzte Hinweise auf systematische Gruppenunterschiede oder Engagement-Relevanz.",
            "Status": "nicht gestützt",
            "Verweis auf Tabelle/Abbildung": "Tabelle 3; Tabelle 5; Abbildung 4",
        },
        {
            "Hypothese": "H6 Kommentare/Topics",
            "Zentrales Ergebnis": "Kommentar-Sentiment unterscheidet sich signifikant zwischen VI und RI und korreliert moderat positiv mit Engagement; Topic-Vielfalt korreliert schwach positiv mit Engagement.",
            "Interpretation": "Kommentar- und Topic-Merkmale zeigen die stärksten engagementbezogenen Hinweise, bleiben aber wegen Topic-Operationalisierung und Videolevel-Aggregation vorsichtig zu interpretieren.",
            "Status": "teilweise gestützt",
            "Verweis auf Tabelle/Abbildung": "Tabelle 3; Tabelle 4; Tabelle 5; Abbildung 5; Abbildung 7",
        },
    ]
    table = pd.DataFrame(rows)
    export_table(table, dirs["tab_main"] / "tab_07_hypothesenpruefung", created)
    export_table(table, dirs["tab_main"] / "tab_07_hypothesenpruefung_template", created)
    overview.append(
        {
            "Kapitel": "6 Diskussion",
            "Typ": "Tabelle 7",
            "Titel": "Zusammenfassende Prüfung der Hypothesen",
            "Datei": "exports/report/tables/tab_07_hypothesenpruefung.csv",
            "Verwendung im Text": "Hypothesenbezug verdichten",
        }
    )


def wrapped(text: str, width: int = 24) -> str:
    return "\n".join(textwrap.wrap(str(text), width=width, break_long_words=False))


def boxplot_panel(df: pd.DataFrame, features: list[tuple[str, str]], title: str, ncols: int = 2) -> plt.Figure:
    n = len(features)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.6 * nrows), squeeze=False)
    for ax, (label, col) in zip(axes.flat, features):
        temp = df[["Influencerinnen-Typ", col]].copy()
        temp[col] = pd.to_numeric(temp[col], errors="coerce")
        temp = temp.dropna()
        sns.boxplot(
            data=temp,
            x="Influencerinnen-Typ",
            y=col,
            order=GROUP_ORDER,
            palette=PALETTE,
            hue="Influencerinnen-Typ",
            hue_order=GROUP_ORDER,
            dodge=False,
            width=0.55,
            showfliers=True,
            ax=ax,
            legend=False,
        )
        ax.set_xlabel("Influencerinnen-Typ")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.set_xticks(range(len(GROUP_ORDER)), GROUP_ORDER)
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    return fig


def create_figures_2_3(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    fig2_specs = ["Videolänge", "Cut Count", "Szenenvielfalt", "Kameradistanzvariation", "Sättigung", "Aesthetic Score"]
    mapping = feature_column_map(df, VISUAL_FEATURES)
    fig2_features = [(label, mapping[label]) for label in fig2_specs if label in mapping]
    if fig2_features:
        fig = boxplot_panel(df, fig2_features, "Verteilung ausgewählter visueller Merkmale nach Influencerinnen-Typ", ncols=3)
        save_figure(fig, dirs["fig_main"] / "abb_02_visuelle_merkmale_boxplots", created)
        overview.append(
            {
                "Kapitel": "5.2 Visuelle Merkmale",
                "Typ": "Abbildung 2",
                "Titel": "Verteilung ausgewählter visueller Merkmale nach Influencerinnen-Typ",
                "Datei": "exports/report/figures/abb_02_visuelle_merkmale_boxplots.png",
                "Verwendung im Text": "Zentrale visuelle Muster zeigen",
            }
        )
    else:
        missing.add("Abbildung 2", ", ".join(fig2_specs), "Plot wurde nicht erstellt")

    fig3_specs = ["Helligkeit", "Kontrast", "Sättigung", "Schärfe"]
    fig3_features = [(label, mapping[label]) for label in fig3_specs if label in mapping]
    if fig3_features:
        fig = boxplot_panel(df, fig3_features, "Verteilung bildtechnischer Merkmale nach Influencerinnen-Typ", ncols=2)
        save_figure(fig, dirs["fig_main"] / "abb_03_bildtechnische_merkmale", created)
        overview.append(
            {
                "Kapitel": "5.2.2 Bildtechnik",
                "Typ": "Abbildung 3",
                "Titel": "Verteilung bildtechnischer Merkmale nach Influencerinnen-Typ",
                "Datei": "exports/report/figures/abb_03_bildtechnische_merkmale.png",
                "Verwendung im Text": "Helligkeit, Kontrast, Sättigung, Schärfe vergleichen",
            }
        )
    else:
        missing.add("Abbildung 3", ", ".join(fig3_specs), "Plot wurde nicht erstellt")


def percent_distribution(df: pd.DataFrame, col: str, top_n: int | None = None) -> pd.DataFrame:
    temp = df[["Influencerinnen-Typ", col]].dropna().copy()
    temp[col] = temp[col].astype(str)
    if top_n:
        keep = set()
        for group in GROUP_ORDER:
            keep.update(temp.loc[temp["Influencerinnen-Typ"] == group, col].value_counts().head(top_n).index)
        temp = temp[temp[col].isin(keep)]
    counts = temp.groupby(["Influencerinnen-Typ", col], observed=True).size().reset_index(name="n")
    totals = temp.groupby("Influencerinnen-Typ", observed=True).size().rename("total")
    counts = counts.merge(totals, on="Influencerinnen-Typ")
    counts["Anteil in %"] = counts["n"] / counts["total"] * 100
    return counts.rename(columns={col: "Label"})


def create_figure_4(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    panels = []
    for title, aliases in [
        ("Caption-Sentiment", ("caption_sentiment_label",)),
        ("Kommentar-Sentiment", ("dominant_comment_sentiment", "comment_sentiment_label")),
    ]:
        col = first_existing(df, aliases)
        if col:
            dist = percent_distribution(df, col)
            panels.append((title, dist))
    if not panels:
        missing.add("Abbildung 4", "Caption-/Kommentar-Sentimentlabels", "Plot wurde nicht erstellt")
        return
    fig, axes = plt.subplots(1, len(panels), figsize=(6.0 * len(panels), 4.2), squeeze=False)
    for ax, (title, dist) in zip(axes.flat, panels):
        sns.barplot(
            data=dist,
            x="Label",
            y="Anteil in %",
            hue="Influencerinnen-Typ",
            hue_order=GROUP_ORDER,
            palette=PALETTE,
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("Sentimentlabel")
        ax.set_ylabel("Anteil in %")
        ticks = ax.get_xticks()
        labels = [wrapped(t.get_text(), 12) for t in ax.get_xticklabels()]
        ax.set_xticks(ticks, labels)
        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles, ["VI", "RI"], title="Influencerinnen-Typ")
    fig.suptitle("Verteilung der Sentimentlabels in Captions und Kommentaren", y=1.02)
    fig.tight_layout()
    save_figure(fig, dirs["fig_main"] / "abb_04_sentiment_captions_kommentare", created)
    overview.append(
        {
            "Kapitel": "5.3 Textuelle Merkmale",
            "Typ": "Abbildung 4",
            "Titel": "Verteilung der Sentimentlabels in Captions und Kommentaren",
            "Datei": "exports/report/figures/abb_04_sentiment_captions_kommentare.png",
            "Verwendung im Text": "Captions vs. Kommentare vergleichen",
        }
    )


def create_figure_5(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    col = first_existing(df, ("dominant_topic_label", "topic_label", "topic_name", "topic"))
    if not col:
        missing.add("Abbildung 5", "dominantes Topic", "Plot wurde nicht erstellt")
        return
    dist = percent_distribution(df, col, top_n=10)
    if dist.empty:
        missing.add("Abbildung 5", col, "Keine Topic-Daten vorhanden")
        return
    order = (
        dist.groupby("Label")["Anteil in %"]
        .max()
        .sort_values(ascending=True)
        .index.tolist()
    )
    dist["Topic"] = dist["Label"].map(lambda x: wrapped(x, 34))
    order_wrapped = [wrapped(x, 34) for x in order]
    fig, ax = plt.subplots(figsize=(10, max(5, 0.42 * len(order_wrapped))))
    sns.barplot(
        data=dist,
        y="Topic",
        x="Anteil in %",
        hue="Influencerinnen-Typ",
        hue_order=GROUP_ORDER,
        order=order_wrapped,
        palette=PALETTE,
        ax=ax,
    )
    ax.set_title("Häufigste Kommentar-Topics nach Influencerinnen-Typ")
    ax.set_xlabel("Anteil in %")
    ax.set_ylabel("Topic")
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles, ["VI", "RI"], title="Influencerinnen-Typ")
    fig.tight_layout()
    save_figure(fig, dirs["fig_main"] / "abb_05_top_topics_kommentare", created)
    overview.append(
        {
            "Kapitel": "5.3.4 Topic Analyse",
            "Typ": "Abbildung 5",
            "Titel": "Häufigste Kommentar-Topics nach Influencerinnen-Typ",
            "Datei": "exports/report/figures/abb_05_top_topics_kommentare.png",
            "Verwendung im Text": "Topic-Unterschiede visualisieren",
        }
    )


def create_figure_6(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    scene_col = first_existing(df, ("scene_unique_labels", "scene_classification__scene_unique_labels"))
    y_col = "log_engagement_rate" if "log_engagement_rate" in df.columns else "engagement_rate" if "engagement_rate" in df.columns else None
    if not scene_col or not y_col:
        missing.add("Abbildung 6", "scene_unique_labels und/oder engagement_rate", "Plot wurde nicht erstellt")
        return
    temp = df[["Influencerinnen-Typ", scene_col, y_col]].copy()
    temp[scene_col] = pd.to_numeric(temp[scene_col], errors="coerce")
    temp[y_col] = pd.to_numeric(temp[y_col], errors="coerce")
    temp = temp.dropna()
    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    sns.scatterplot(
        data=temp,
        x=scene_col,
        y=y_col,
        hue="Influencerinnen-Typ",
        hue_order=GROUP_ORDER,
        palette=PALETTE,
        alpha=0.75,
        ax=ax,
    )
    for group in GROUP_ORDER:
        subset = temp[temp["Influencerinnen-Typ"] == group]
        if len(subset) >= 5 and subset[scene_col].nunique() > 1:
            sns.regplot(data=subset, x=scene_col, y=y_col, scatter=False, ax=ax, color=PALETTE[group], ci=None)
    ax.set_title("Zusammenhang zwischen Szenenvielfalt und Engagement-Rate")
    ax.set_xlabel("Anzahl unterschiedlicher Szenenlabels")
    ax.set_ylabel("log1p(Engagement-Rate)" if y_col == "log_engagement_rate" else "Engagement-Rate")
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles, ["VI", "RI"], title="Influencerinnen-Typ")
    fig.tight_layout()
    save_figure(fig, dirs["fig_main"] / "abb_06_szenenvielfalt_engagement", created)
    overview.append(
        {
            "Kapitel": "5.4 Engagement",
            "Typ": "Abbildung 6",
            "Titel": "Zusammenhang zwischen Szenenvielfalt und Engagement-Rate",
            "Datei": "exports/report/figures/abb_06_szenenvielfalt_engagement.png",
            "Verwendung im Text": "Explorativen Zusammenhang zwischen Szenenvariation und Engagement zeigen",
        }
    )


def create_figure_7(corr_table: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    if corr_table.empty or "Spearman ρ" not in corr_table.columns:
        missing.add("Abbildung 7", "Tabelle 5/Korrelationen", "Plot wurde nicht erstellt")
        return
    plot_df = corr_table.dropna(subset=["Spearman ρ"]).copy()
    if plot_df.empty:
        missing.add("Abbildung 7", "berechnete Spearman-Korrelationen", "Plot wurde nicht erstellt")
        return
    plot_df = plot_df.sort_values("Spearman ρ")
    if len(plot_df) > 18:
        low = plot_df.head(9)
        high = plot_df.tail(9)
        plot_df = pd.concat([low, high]).drop_duplicates("Feature")
    colors = [COLOR_VI if value >= 0 else COLOR_RI for value in plot_df["Spearman ρ"]]
    fig, ax = plt.subplots(figsize=(8.2, max(5, 0.35 * len(plot_df))))
    ax.barh(plot_df["Feature"], plot_df["Spearman ρ"], color=colors)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.set_title("Spearman-Korrelationen zentraler Merkmale mit der Engagement-Rate")
    ax.set_xlabel("Spearman ρ")
    ax.set_ylabel("Feature")
    fig.tight_layout()
    save_figure(fig, dirs["fig_main"] / "abb_07_spearman_korrelationen_engagement", created)
    overview.append(
        {
            "Kapitel": "5.4 Engagement",
            "Typ": "Abbildung 7",
            "Titel": "Spearman-Korrelationen zentraler Merkmale mit der Engagement-Rate",
            "Datei": "exports/report/figures/abb_07_spearman_korrelationen_engagement.png",
            "Verwendung im Text": "Engagement-Muster sichtbar machen",
        }
    )


def create_figure_8(human: pd.DataFrame, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    if human.empty or "mean_rating" not in human.columns:
        missing.add("Abbildung 8", "Human-Evaluation-Mittelwerte", "Plot wurde nicht erstellt")
        return
    plot_df = human.copy()
    plot_df["Kategorie"] = plot_df.get("feature", plot_df.get("feature_group", "Kategorie")).astype(str).map(lambda x: wrapped(x, 28))
    plot_df = plot_df.sort_values("mean_rating", ascending=True)
    fig, ax = plt.subplots(figsize=(8.5, max(5.5, 0.38 * len(plot_df))))
    xerr = plot_df["sd_rating"] if "sd_rating" in plot_df.columns else None
    bars = ax.barh(
        plot_df["Kategorie"],
        plot_df["mean_rating"],
        xerr=xerr,
        color=COLOR_HUMAN_EVAL_PRIMARY,
        alpha=0.9,
        error_kw={"ecolor": "#222222", "elinewidth": 1.2, "capsize": 3, "capthick": 1.2},
    )
    if bars.errorbar is not None:
        for capline in bars.errorbar.lines[1]:
            capline.set_markerfacecolor("#222222")
            capline.set_markeredgecolor("#222222")
    ax.set_title("Human Evaluation der automatisierten Annotationen")
    ax.set_xlabel("Mittlere menschliche Bewertung (1 = niedrig, 5 = hoch)")
    ax.set_ylabel("Merkmalskategorie")
    ax.set_xlim(0, 5)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.text(
        0.99,
        0.02,
        "Schwarze Linien: ±1 Standardabweichung",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9,
        color="#333333",
    )
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    save_figure(fig, dirs["fig_main"] / "abb_08_human_evaluation_modellbewertung", created)
    overview.append(
        {
            "Kapitel": "5.1 Modellvalidierung",
            "Typ": "Abbildung 8",
            "Titel": "Bewertung der automatisierten Annotationen in der Human Evaluation",
            "Datei": "exports/report/figures/abb_08_human_evaluation_modellbewertung.png",
            "Verwendung im Text": "Nach Tabelle 6 einfügen",
        }
    )


def create_figure_9(root: Path, dirs: dict[str, Path], created: list[Path], overview: list[dict[str, str]], missing: MissingReport) -> None:
    path = root / "data/04_analysis_results/human_evaluation/human_eval_reliability_overall_feature.csv"
    if not path.exists():
        missing.add("Abbildung 9", "human_eval_reliability_overall_feature.csv", "Plot wurde nicht erstellt")
        return
    reliability = pd.read_csv(path)
    required = {"feature", "exact_agreement", "within_one_agreement", "weighted_kappa_quadratic"}
    missing_cols = required - set(reliability.columns)
    if missing_cols:
        missing.add("Abbildung 9", ", ".join(sorted(missing_cols)), "Plot wurde nicht erstellt")
        return

    plot_df = reliability.copy()
    if "is_overall" in plot_df.columns:
        plot_df = plot_df[plot_df["is_overall"] == False].copy()  # noqa: E712
    plot_df = plot_df.dropna(subset=["exact_agreement", "within_one_agreement"])
    if plot_df.empty:
        missing.add("Abbildung 9", "Interrater-Übereinstimmungswerte", "Plot wurde nicht erstellt")
        return

    plot_df["Feature"] = plot_df["feature"].astype(str).map(lambda x: wrapped(x, 28))
    plot_df = plot_df.sort_values("within_one_agreement", ascending=True)
    y = np.arange(len(plot_df))
    height = 0.36

    fig, ax = plt.subplots(figsize=(9.2, max(6.0, 0.44 * len(plot_df))))
    ax.barh(
        y - height / 2,
        plot_df["within_one_agreement"],
        height=height,
        color=COLOR_HUMAN_EVAL_PRIMARY,
        alpha=0.90,
        label="max. 1 Skalenpunkt Unterschied",
    )
    ax.barh(
        y + height / 2,
        plot_df["exact_agreement"],
        height=height,
        color=COLOR_HUMAN_EVAL_SECONDARY,
        alpha=0.90,
        label="exakt gleiches Rating",
    )

    for yi, row in enumerate(plot_df.itertuples(index=False)):
        ax.text(
            min(float(row.within_one_agreement) + 0.018, 1.02),
            yi - height / 2,
            f"{row.within_one_agreement:.0%}",
            va="center",
            ha="left",
            fontsize=8.5,
        )
        ax.text(
            min(float(row.exact_agreement) + 0.018, 1.02),
            yi + height / 2,
            f"{row.exact_agreement:.0%}",
            va="center",
            ha="left",
            fontsize=8.5,
        )
        kappa = getattr(row, "weighted_kappa_quadratic", np.nan)
        kappa_label = "κw = n/a" if pd.isna(kappa) else f"κw = {kappa:.2f}"
        ax.text(1.08, yi, kappa_label, va="center", ha="left", fontsize=8.5, color="#333333")

    ax.axvline(0.80, color="#666666", linewidth=1, linestyle=":", alpha=0.75)
    ax.text(0.805, len(plot_df) - 0.2, "80%", ha="left", va="bottom", fontsize=8.5, color="#555555")
    ax.set_yticks(y, plot_df["Feature"])
    ax.set_xlim(0, 1.20)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}" if value <= 1 else ""))
    ax.set_xlabel("Übereinstimmung zwischen den zwei Annotator:innen")
    ax.set_ylabel("Merkmalskategorie")
    ax.set_title("Interrater-Übereinstimmung der Human Evaluation")
    ax.legend(title="Übereinstimmungsmaß", loc="lower right", frameon=True)
    ax.text(
        0.0,
        -0.12,
        "κw = quadratisch gewichtetes Cohen's Kappa; höhere Prozentwerte bedeuten stärkere praktische Übereinstimmung.",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        color="#333333",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    save_figure(fig, dirs["fig_main"] / "abb_09_interrater_uebereinstimmung", created)
    overview.append(
        {
            "Kapitel": "5.1 Modellvalidierung",
            "Typ": "Abbildung 9",
            "Titel": "Interrater-Übereinstimmung der Human Evaluation",
            "Datei": "exports/report/figures/abb_09_interrater_uebereinstimmung.png",
            "Verwendung im Text": "Vor oder nach Tabelle 6 zur Reliabilität der Human Evaluation einfügen",
        }
    )


def create_appendix_plots(df: pd.DataFrame, dirs: dict[str, Path], created: list[Path], missing: MissingReport) -> None:
    specs = [*VISUAL_FEATURES, *TEXT_FEATURES]
    mapping = feature_column_map(df, specs)
    numeric_cols = [(label, col) for label, col in mapping.items() if pd.to_numeric(df[col], errors="coerce").notna().sum() >= 5]
    if "engagement_rate" in df.columns:
        numeric_cols.append(("Engagement-Rate", "engagement_rate"))

    if len(numeric_cols) >= 3:
        corr_df = pd.DataFrame({label: pd.to_numeric(df[col], errors="coerce") for label, col in numeric_cols})
        corr = corr_df.corr(method="spearman")
        fig, ax = plt.subplots(figsize=(max(9, 0.55 * len(corr)), max(7, 0.45 * len(corr))))
        sns.heatmap(corr, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax, square=False)
        ax.set_title("Vollständige Spearman-Korrelationsheatmap")
        fig.tight_layout()
        save_figure(fig, dirs["fig_appendix"] / "app_korrelationsheatmap", created)

    if numeric_cols:
        fig = boxplot_panel(df, numeric_cols, "Detailplots für alle numerischen Features", ncols=3)
        save_figure(fig, dirs["fig_appendix"] / "app_detailplots_alle_numerischen_features", created)

    topic_col = first_existing(df, ("dominant_topic_label", "topic_label", "topic_name", "topic"))
    if topic_col:
        dist = percent_distribution(df, topic_col, top_n=25)
        if not dist.empty:
            order = dist.groupby("Label")["Anteil in %"].max().sort_values(ascending=True).index.tolist()
            dist["Topic"] = dist["Label"].map(lambda x: wrapped(x, 36))
            fig, ax = plt.subplots(figsize=(10, max(6, 0.35 * len(order))))
            sns.barplot(
                data=dist,
                y="Topic",
                x="Anteil in %",
                hue="Influencerinnen-Typ",
                hue_order=GROUP_ORDER,
                order=[wrapped(x, 36) for x in order],
                palette=PALETTE,
                ax=ax,
            )
            ax.set_title("Vollständige Topic-Verteilungen")
            ax.set_xlabel("Anteil in %")
            ax.set_ylabel("Topic")
            handles, _ = ax.get_legend_handles_labels()
            ax.legend(handles, ["VI", "RI"], title="Influencerinnen-Typ")
            fig.tight_layout()
            save_figure(fig, dirs["fig_appendix"] / "app_topic_verteilungen", created)

    if "engagement_rate" in df.columns:
        fig = boxplot_panel(df, [("Engagement-Rate", "engagement_rate")], "Verteilung der Engagement-Rate nach Gruppe", ncols=1)
        save_figure(fig, dirs["fig_appendix"] / "app_engagement_rate_nach_gruppe", created)
        influencer_col = first_existing(df, ("influencer_clean", "influencer", "author_username"))
        if influencer_col:
            temp = df[[influencer_col, "engagement_rate", "Influencerinnen-Typ"]].copy()
            temp["engagement_rate"] = pd.to_numeric(temp["engagement_rate"], errors="coerce")
            temp = temp.dropna()
            if not temp.empty:
                order = temp.groupby(influencer_col)["engagement_rate"].median().sort_values().index.tolist()
                fig, ax = plt.subplots(figsize=(10, max(5, 0.3 * len(order))))
                sns.boxplot(
                    data=temp,
                    y=influencer_col,
                    x="engagement_rate",
                    hue="Influencerinnen-Typ",
                    hue_order=GROUP_ORDER,
                    order=order,
                    palette=PALETTE,
                    ax=ax,
                )
                ax.set_title("Engagement-Rate nach Influencerin")
                ax.set_xlabel("Engagement-Rate")
                ax.set_ylabel("Influencerin")
                handles, _ = ax.get_legend_handles_labels()
                ax.legend(handles, ["VI", "RI"], title="Influencerinnen-Typ")
                fig.tight_layout()
                save_figure(fig, dirs["fig_appendix"] / "app_engagement_rate_nach_influencerin", created)

    for title, aliases, filename in APPENDIX_LABEL_PLOTS:
        col = first_existing(df, aliases)
        if not col:
            continue
        dist = percent_distribution(df, col, top_n=20)
        if dist.empty:
            continue
        order = dist.groupby("Label")["Anteil in %"].max().sort_values(ascending=True).index.tolist()
        dist["LabelWrapped"] = dist["Label"].map(lambda x: wrapped(x, 28))
        fig, ax = plt.subplots(figsize=(8.5, max(4.5, 0.35 * len(order))))
        sns.barplot(
            data=dist,
            y="LabelWrapped",
            x="Anteil in %",
            hue="Influencerinnen-Typ",
            hue_order=GROUP_ORDER,
            order=[wrapped(x, 28) for x in order],
            palette=PALETTE,
            ax=ax,
        )
        ax.set_title(title + " nach Influencerinnen-Typ")
        ax.set_xlabel("Anteil in %")
        ax.set_ylabel(title)
        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles, ["VI", "RI"], title="Influencerinnen-Typ")
        fig.tight_layout()
        save_figure(fig, dirs["fig_appendix"] / filename, created)


def main() -> None:
    root = find_project_root()
    dirs = setup_dirs(root)
    configure_style()
    missing = MissingReport()
    created: list[Path] = []
    overview: list[dict[str, str]] = []

    master_path = choose_master_file(root)
    df = pd.read_csv(master_path)
    df = normalize_video_id(df)
    df = add_group_column(df, missing)
    df = merge_text_results(root, df, missing)
    df = add_derived_columns(df, missing)

    create_systematic_reporting_outputs(df, dirs, created, missing)

    create_table_1(df, dirs, created, overview)
    create_table_2(df, dirs, created, overview, missing)
    create_table_3(df, dirs, created, overview, missing)
    create_table_4(df, dirs, created, overview, missing)
    corr_table = create_table_5(df, dirs, created, overview, missing)
    human = create_table_6(root, dirs, created, overview, missing)
    create_table_7(dirs, created, overview)

    create_figures_2_3(df, dirs, created, overview, missing)
    create_figure_4(df, dirs, created, overview, missing)
    create_figure_5(df, dirs, created, overview, missing)
    create_figure_6(df, dirs, created, overview, missing)
    create_figure_7(corr_table, dirs, created, overview, missing)
    create_figure_8(human, dirs, created, overview, missing)
    create_figure_9(root, dirs, created, overview, missing)
    create_appendix_plots(df, dirs, created, missing)

    print(f"Zentrale Ergebnisdatei: {rel(master_path, root)}")
    print(f"Erzeugte Dateien: {len(created)}")
    print(f"Fehlende/übersprungene Variablen: {len(missing.rows)}")



## Ausführung


In [ ]:
main()
